# Optimization analytics — reverse-ETL from Fabric back to Cosmos

**Cosmos DB** is the *operational* store for the live travel agent — low-latency reads/writes
on the request path. **Mirroring** streams it into **Microsoft Fabric** (the *analytical* plane)
with no ETL pipeline to build. This notebook runs the kind of heavy, cross-session analysis you
would **never** run on the transactional path, then **reverse-ETLs** each result back into Cosmos
(`OptimizationInsights`) so the app, the **Optimization Console**, and **Power BI** can *act* on it.

```
Cosmos (operational) --mirror--> Fabric (Spark analysis) --reverse-ETL--> Cosmos OptimizationInsights --> app / Console / Power BI act
```

That closed loop is the substrate for **Level 4 (autonomous)** and **Level 5 (adaptive)**
optimization: Fabric computes the intelligence, reverse-ETL lands it where the operational
system can use it in real time.

**What this notebook computes** — each analysis reverse-ETLs flat rows (keyed by `type`) that the
Console and the Power BI report read directly:
- **Conversion funnel + abandonment cause** (Sections 2–3) — where sessions leak and *why* they don't convert.
- **Measured saving** (Section 4b) — keyed by optimization: the **model-selection counterfactual** (each captured turn re-priced under the model it actually ran on vs. the all-premium baseline) **and** the **memory-retention** telemetry saving (input tokens recalls avoided by dropping pruned memories, aggregated from the mirrored `ApiEvents`; `$0` until the policy is applied). The honest dollar figures behind the estimate cards.
- **Agent-path cost concentration** (5b) — which `agent_path`s dominate token cost (where model tiering + tool-dedup pay off first).
- **Turn metrics** (5c) — the Console KPIs: total turns/tokens, estimated cost, trivial-turn share, model distribution, cost-by-tier.
- **Agent scorecard** (5d) — per-agent health across **cost efficiency**, **model selection**, and **workflow efficiency**, rolled up from the mirrored `NodeExecutions` node-grain (feeds the Power BI **Agent Performance** page 6b).
- **Memory intelligence** (6) — salience / health / supersession of the memory subsystem.
- **LLM analyst** (7) — the model *proposes* one recommendation card **per positive detected opportunity** (**model-selection** + **tool-call-dedup**) and the engine's **guardrails** *dispose*: bind it to the exact declared opportunity seam, require citations, and override its dollar figure with the engine-measured saving. Each accepted card is reverse-ETL'd as a `discovered_opportunity` **and** a flat `recommendation_card` row — now including a compact **evidence line** and a **caveat** — so the Console and the Power BI **recommendation gallery** show the same proof, limitation, and `governed_state` badge. Detections with zero candidates/turns or zero saving overwrite their deterministic report IDs as `suppressed_opportunity` tombstones, preventing stale cards.

**You implement two pieces** (everything else is provided):
- **TODO 1** — classify *why* a session didn't convert (the analytics decision you own; the notebook analog of `classify_complexity_tier`).
- **TODO 2** — the **reverse-ETL write** back to Cosmos (the pattern this module teaches).

## 0. Load the Cosmos connector (run this first)

The reverse-ETL write (Section 5) uses the **Azure Cosmos DB Spark connector** (`cosmos.oltp`), which isn't in Fabric's default Spark runtime. Run the cell below **first** — it loads the connector plus the Fabric auth library and **restarts the Spark session** (takes ~1 minute). After it finishes, run the **Parameters** cell, then continue top to bottom.

In [ ]:
%%configure -f
{
    "conf": {
        "spark.jars.packages": "com.azure.cosmos.spark:azure-cosmos-spark_3-5_2-12:4.41.0,com.azure.cosmos.spark:fabric-cosmos-spark-auth_3:1.1.0"
    }
}

In [ ]:
# --- Parameters (overridable via RunNotebook parameterValues) ---
# Cosmos (reverse-ETL write target — the OPERATIONAL store)
COSMOS_ENDPOINT = ""            # https://<account>.documents.azure.com:443/
COSMOS_DATABASE = "TravelAssistant"
INSIGHTS_CONTAINER = "OptimizationInsights"
TENANT_ID = ""                 # Entra tenant (for the Fabric->Cosmos AAD write)
# Mirror SQL analytics endpoint (host, no port) + database (mirror artifact name)
SQL_EP = ""
SQL_DB = ""
SOURCE_SCHEMA = COSMOS_DATABASE   # the mirror schema == the Cosmos DB name
DEMO_TENANTS = ("analytics", "marvel")
TENANT = "analytics"            # one demo tenant per Fabric notebook activity
RUN_GLOBAL_INSIGHTS = True      # True for analytics; False for the dependent marvel activity
# Azure OpenAI (the LLM analyst in Section 7 — keyless/AAD, the same account the app uses).
# Leave blank to skip the live call: the analyst then falls back to the deterministic
# proposer (engine parity), so the section still reverse-ETLs a guardrailed card.
AOAI_ENDPOINT = ""              # https://<account>.openai.azure.com/
AOAI_DEPLOYMENT = "gpt-5.1"
AOAI_API_VERSION = "2025-04-01-preview"

In [ ]:
# --- Normalize and validate pipeline-injected parameter overrides ---
TENANT = str(TENANT).strip().lower()
if TENANT not in DEMO_TENANTS:
    raise ValueError(f"TENANT must be one of {DEMO_TENANTS}; got {TENANT!r}")
if isinstance(RUN_GLOBAL_INSIGHTS, bool):
    pass
elif isinstance(RUN_GLOBAL_INSIGHTS, str):
    _bool_values = {
        "1": True, "true": True, "yes": True, "y": True, "on": True,
        "0": False, "false": False, "no": False, "n": False, "off": False,
    }
    _bool_key = RUN_GLOBAL_INSIGHTS.strip().lower()
    if _bool_key not in _bool_values:
        raise ValueError(
            "RUN_GLOBAL_INSIGHTS must be a boolean or one of "
            f"{tuple(_bool_values)}; got {RUN_GLOBAL_INSIGHTS!r}"
        )
    RUN_GLOBAL_INSIGHTS = _bool_values[_bool_key]
elif isinstance(RUN_GLOBAL_INSIGHTS, int) and RUN_GLOBAL_INSIGHTS in (0, 1):
    RUN_GLOBAL_INSIGHTS = bool(RUN_GLOBAL_INSIGHTS)
else:
    raise ValueError(
        "RUN_GLOBAL_INSIGHTS must be a boolean, 0/1, or a recognized boolean string; "
        f"got {RUN_GLOBAL_INSIGHTS!r}"
    )
print(f"Notebook run: tenant={TENANT}; global insights={RUN_GLOBAL_INSIGHTS}")

### Run both demo tenants safely

This notebook is a **single-tenant worker**, not a self-recursive controller. Phase 2 of
`provision_fabric.py` creates or updates the `RefreshAllDemoTenants` Fabric Data Pipeline with:

1. `analytics`: `TENANT=analytics`, `RUN_GLOBAL_INSIGHTS=true`
2. `marvel` (depends on `analytics` succeeding): `TENANT=marvel`, `RUN_GLOBAL_INSIGHTS=false`

The first activity writes the shared `_global_optimizations` and `_global_memory` rows once. The
second writes only Marvel's tenant-scoped rows. This supported two-activity pattern is deterministic,
retryable per tenant, and avoids duplicate global memory analysis.

## 1. Read the mirror
The analytical plane reads the **mirrored** tables through the SQL endpoint — the operational Cosmos account is never touched by analytics.

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
now = datetime.now(timezone.utc).isoformat()

# ---- read mirrored tables via the SQL analytics endpoint (the analytical plane
# reads the MIRROR, never the transactional Cosmos account) ----
_sql_token = mssparkutils.credentials.getToken("pbi")
_jdbc = f"jdbc:sqlserver://{SQL_EP}:1433;database={SQL_DB};encrypt=true;trustServerCertificate=false"

def read_sql(table, tenant_scoped=True):
    df = (spark.read.format("jdbc")
          .option("url", _jdbc)
          .option("dbtable", f"[{SOURCE_SCHEMA}].[{table}]")
          .option("accessToken", _sql_token)
          .load())
    return df.where(F.col("tenantId") == TENANT) if tenant_scoped and "tenantId" in df.columns else df

turns = read_sql("OptimizationTurns")
# schema-compat: pre-rename telemetry stored the model tier as `model_tier`; the current
# app/seed write `complexity_tier` (same field). Normalize so downstream (5c) always finds it.
if "complexity_tier" not in turns.columns and "model_tier" in turns.columns:
    turns = turns.withColumnRenamed("model_tier", "complexity_tier")
trips = read_sql("Trips")
messages = read_sql("Messages")
policies = read_sql("OptimizationPolicies", tenant_scoped=False)
governance = read_sql("OptimizationGovernance")
print("turns:", turns.count(), "trips:", trips.count(), "messages:", messages.count(),
      "policies:", policies.count(), "governance:", governance.count())

In [ ]:
# ---- canonical controlled Demo 4 source evaluator ----
# The app evaluator is embedded verbatim at generation time because Fabric cannot import the
# application package. This keeps one evaluator/result schema while the adapter only supplies
# rows captured from the same mirrored DirectQuery snapshot.
CONTROLLED_DEMO4_EMBEDDED_PRICING = {'gpt-5.1': {'input': 1.25, 'output': 10.0}, 'gpt-5-mini': {'input': 0.25, 'output': 2.0}, 'gpt-5-nano': {'input': 0.05, 'output': 0.4}}
CONTROLLED_DEMO4_EMBEDDED_PRICING_SOURCE = r"python\data\model_pricing.json"
CONTROLLED_DEMO4_EMBEDDED_PRICING_SHA256 = "d6d43d2eb81ab3cc6c17d01c6b109576cac7e121e75efc0ecf225cc32d06c493"
exec('"""Deterministic source fixture for the controlled Demo 4 baseline."""\n\nfrom __future__ import annotations\n\nimport copy\nimport hashlib\nimport json\nfrom collections import Counter\nfrom datetime import datetime, timedelta, timezone\nfrom decimal import Decimal, InvalidOperation\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom typing import Any\n\nFIXTURE_VERSION = "controlled-demo4-v1"\nDEFAULT_ANCHOR = "2026-09-24T12:00:00Z"\nTURN_COUNT = 200\nSESSION_COUNT = 192\nCONFIRMED_TRIP_COUNT = 56\nWINDOW_MINUTES = 20\nCANONICAL_MINUTE_PROFILE = (\n    5, 6, 8, 9, 10, 12, 14, 16, 15, 13,\n    11, 10, 11, 13, 12, 10, 8, 7, 6, 4,\n)\nBURST_VERSION = "controlled-demo4-after-v1"\nBURST_ANCHOR = "2026-09-24T13:00:00Z"\nPRICING_VERSION = "model-pricing-v1"\nROUNDING_TOLERANCE_USD = 0.00005\nROUNDING_TOLERANCE_PCT = 0.05\nEVALUATION_FIELDS = (\n    "tenant_id",\n    "policy_status",\n    "dataset_phase",\n    "display_state",\n    "state_valid",\n    "state_reason",\n    "measurement_status",\n    "fixture_version",\n    "burst_version",\n    "burst_anchor",\n    "burst_window_minutes",\n    "baseline_expected_count",\n    "baseline_observed_count",\n    "baseline_expected_ids",\n    "baseline_observed_ids",\n    "burst_expected_count",\n    "burst_observed_count",\n    "burst_expected_ids",\n    "burst_observed_ids",\n    "model_counts",\n    "measurement",\n)\nMEASUREMENT_FIELDS = (\n    "measurement_status",\n    "measurement_kind",\n    "measurement_scope",\n    "measurement_tenant",\n    "fixture_version",\n    "burst_version",\n    "burst_anchor",\n    "burst_window_minutes",\n    "expected_count",\n    "observed_count",\n    "observed_ids",\n    "manifest_identity",\n    "model_counts",\n    "pricing_source",\n    "pricing_version",\n    "pricing_sha256",\n    "validation_status",\n    "validation_reason",\n    "rounding_tolerance_usd",\n    "rounding_tolerance_pct",\n    "baseline_cost_usd_unrounded",\n    "actual_cost_usd_unrounded",\n    "saving_usd_unrounded",\n    "saving_pct_unrounded",\n    "baseline_cost_usd",\n    "actual_cost_usd",\n    "saving_usd",\n    "saving_pct",\n    "positive_measured_result",\n)\nTENANTS = ("analytics", "marvel")\nCOHORT_CONTAINERS = (\n    "OptimizationTurns",\n    "Debug",\n    "NodeExecutions",\n    "Sessions",\n    "Messages",\n    "Trips",\n)\nDERIVED_CONTAINERS = ("OptimizationInsights", "OptimizationGovernance")\nCONTROLLED_DERIVED_TYPES = frozenset({\n    "agent_opportunity",\n    "controlled_demo4_derived",\n    "controlled_demo4_state",\n    "recommendation_card",\n    "turn_metrics",\n    "funnel_stage",\n    "conversion_kpi",\n    "abandonment_cause",\n    "agent_path_cost",\n    "memory_retention",\n    "agent_scorecard",\n    "memory_kpi",\n    "memory_type",\n    "memory_salience",\n    "memory_health",\n    "optimization_result",\n    "slo_metric",\n    "slo_policy",\n})\nPREMIUM_DEPLOYMENT = "gpt-5.1"\nPREMIUM_MODEL = "gpt-5.1-2025-11-13"\nBURST_PROFILES = {\n    "complex": {\n        "weight": 0.35,\n        "deployment": PREMIUM_DEPLOYMENT,\n        "model": PREMIUM_MODEL,\n    },\n    "routine": {\n        "weight": 0.55,\n        "deployment": "gpt-5-mini",\n        "model": "gpt-5-mini-2025-08-07",\n    },\n    "trivial": {\n        "weight": 0.10,\n        "deployment": "gpt-5-nano",\n        "model": "gpt-5-nano-2025-08-07",\n    },\n}\nOUTCOME_COUNTS = {\n    "converted": 56,\n    "city_friction": 52,\n    "cart_abandon": 36,\n    "no_results": 24,\n    "search_stall": 16,\n    "no_engagement": 8,\n}\nEXPECTED_FUNNEL = {\n    "engaged": 192,\n    "searched": 184,\n    "planned": 92,\n    "confirmed": 56,\n}\n\n_IDENTITY_FIELDS = {\n    "id",\n    "tenantId",\n    "userId",\n    "sessionId",\n    "turnId",\n    "debugLogId",\n    "messageId",\n    "tripId",\n}\n_TURN_SPECIALIZED_FIELDS = {\n    "fixture_version",\n    "fixture_ordinal",\n    "burst_version",\n    "controlled_namespace",\n    "burst_ordinal",\n    "burst_anchor",\n    "burst_count",\n    "burst_window_minutes",\n    "complexity_tier",\n    "model_deployment",\n    "model_name",\n    "input_tokens",\n    "output_tokens",\n}\n\n\ndef controlled_fixture_session_prefix(\n    tenant: str,\n    *,\n    fixture_version: str = FIXTURE_VERSION,\n) -> str:\n    """Return the canonical session prefix owned by one baseline fixture tenant."""\n    return f"{fixture_version}-{tenant}-session-"\n\n\ndef controlled_trip_session_prefixes(\n    tenant: str,\n    *,\n    fixture_version: str = FIXTURE_VERSION,\n    burst_version: str = BURST_VERSION,\n) -> tuple[str, str]:\n    """Return every controlled Demo 4 session prefix that can own Trip rows."""\n    return (\n        controlled_fixture_session_prefix(tenant, fixture_version=fixture_version),\n        f"{burst_version}-{tenant}-session-",\n    )\n\n\ndef is_controlled_trip(\n    row: dict[str, Any],\n    tenant: str,\n    *,\n    fixture_version: str = FIXTURE_VERSION,\n    burst_version: str = BURST_VERSION,\n) -> bool:\n    """Whether a Trip is an exact controlled fixture row, not merely a prefix collision."""\n    return is_controlled_fixture_row(\n        "Trips",\n        row,\n        tenant,\n        fixture_version=fixture_version,\n        burst_version=burst_version,\n    )\n\n\n@lru_cache(maxsize=None)\ndef _controlled_ids(\n    container_name: str,\n    tenant: str,\n    fixture_version: str,\n    burst_version: str,\n) -> tuple[frozenset[str], frozenset[str]]:\n    baseline = build_tenant_manifest(tenant, fixture_version=fixture_version)\n    baseline_ids = frozenset(str(row["id"]) for row in baseline[container_name])\n    burst_ids: frozenset[str] = frozenset()\n    if tenant == "analytics":\n        burst = build_after_burst_manifest(\n            tenant=tenant,\n            fixture_version=fixture_version,\n            burst_version=burst_version,\n        )\n        burst_ids = frozenset(str(row["id"]) for row in burst[container_name])\n    return baseline_ids, burst_ids\n\n\ndef is_controlled_fixture_row(\n    container_name: str,\n    row: dict[str, Any],\n    tenant: str,\n    *,\n    fixture_version: str = FIXTURE_VERSION,\n    burst_version: str = BURST_VERSION,\n) -> bool:\n    """Return true only for an exact, fully namespaced controlled fixture row."""\n    if tenant not in TENANTS or row.get("tenantId") != tenant:\n        return False\n    if container_name in DERIVED_CONTAINERS:\n        return (\n            row.get("fixture_version") == fixture_version\n            and row.get("controlled_namespace")\n            == f"{fixture_version}:derived:{tenant}"\n            and row.get("type") in CONTROLLED_DERIVED_TYPES\n        )\n    if container_name not in COHORT_CONTAINERS:\n        return False\n    baseline_ids, burst_ids = _controlled_ids(\n        container_name, tenant, fixture_version, burst_version\n    )\n    row_id = str(row.get("id") or "")\n    if row_id in baseline_ids:\n        return (\n            row.get("fixture_version") == fixture_version\n            and row.get("burst_version") in (None, "")\n        )\n    if row_id in burst_ids:\n        return (\n            row.get("fixture_version") == fixture_version\n            and row.get("burst_version") == burst_version\n            and row.get("controlled_namespace")\n            == f"{fixture_version}:{burst_version}:{tenant}"\n        )\n    return False\n\n\ndef parse_anchor(value: str) -> datetime:\n    """Parse a minute-aligned UTC fixture anchor."""\n    if not isinstance(value, str) or not value.endswith("Z"):\n        raise ValueError("anchor must be an ISO-8601 UTC value ending in Z")\n    try:\n        parsed = datetime.fromisoformat(value[:-1] + "+00:00")\n    except ValueError as exc:\n        raise ValueError("anchor must be a valid ISO-8601 UTC value") from exc\n    if parsed.tzinfo != timezone.utc:\n        raise ValueError("anchor must use UTC")\n    if parsed.second or parsed.microsecond:\n        raise ValueError("anchor must be aligned to a minute")\n    return parsed\n\n\ndef _iso(value: datetime) -> str:\n    return value.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")\n\n\ndef _canonical_turn(ordinal: int) -> dict[str, Any]:\n    input_tokens = 9000 + ((ordinal * 137) % 12000)\n    output_tokens = 350 + ((ordinal * 29) % 900)\n    cached_tokens = input_tokens * (60 + ordinal % 25) // 100\n    return {\n        "ordinal": ordinal,\n        "input_tokens": input_tokens,\n        "output_tokens": output_tokens,\n        "total_tokens": input_tokens + output_tokens,\n        "cached_tokens": cached_tokens,\n        "destination": ("Paris", "Tokyo", "Seattle", "London")[ordinal % 4],\n    }\n\n\nCONTROLLED_DEMO4_V1 = tuple(_canonical_turn(ordinal) for ordinal in range(TURN_COUNT))\n\n\ndef _split_int(total: int, count: int) -> list[int]:\n    quotient, remainder = divmod(total, count)\n    return [quotient + (1 if index < remainder else 0) for index in range(count)]\n\n\ndef _session_for_turn(turn_ordinal: int) -> tuple[int, int]:\n    if turn_ordinal < 16:\n        return turn_ordinal // 2, turn_ordinal % 2\n    return turn_ordinal - 8, 0\n\n\ndef _outcome_for_session(session_ordinal: int) -> str:\n    offset = 0\n    for outcome, count in OUTCOME_COUNTS.items():\n        offset += count\n        if session_ordinal < offset:\n            return outcome\n    raise ValueError(f"session ordinal outside controlled fixture: {session_ordinal}")\n\n\ndef _journey_for_turn(\n    outcome: str,\n    turn_index: int,\n    multi_turn: bool,\n) -> tuple[tuple[str, ...], str]:\n    if outcome == "converted":\n        agents = (\n            ("supervisor", "find_places")\n            if multi_turn and turn_index == 0\n            else ("supervisor", "find_places", "itinerary_generator")\n        )\n        return agents, "Your itinerary is ready and the trip is confirmed."\n    if outcome == "city_friction":\n        return ("supervisor", "find_places"), "Which city is that hotel in?"\n    if outcome == "cart_abandon":\n        return (\n            "supervisor",\n            "find_places",\n            "itinerary_generator",\n        ), "Here is your complete itinerary. Shall I book it?"\n    if outcome == "no_results":\n        return ("supervisor", "find_places"), "I couldn\'t find any matching places for that search."\n    if outcome == "search_stall":\n        return ("supervisor", "find_places"), "I found several options. Tell me how you want to continue."\n    return ("supervisor",), "Happy to help. Where would you like to go?"\n\n\ndef build_tenant_manifest(\n    tenant: str,\n    *,\n    fixture_version: str = FIXTURE_VERSION,\n    anchor: str = DEFAULT_ANCHOR,\n    count: int = TURN_COUNT,\n) -> dict[str, list[dict[str, Any]]]:\n    """Build the exact six-container baseline manifest for one controlled tenant."""\n    if tenant not in TENANTS:\n        raise ValueError(f"unsupported controlled tenant: {tenant}")\n    if fixture_version != FIXTURE_VERSION:\n        raise ValueError(f"unsupported fixture version: {fixture_version}")\n    if count != TURN_COUNT:\n        raise ValueError(f"controlled fixture count must be exactly {TURN_COUNT}")\n    anchor_dt = parse_anchor(anchor)\n\n    manifest = {name: [] for name in COHORT_CONTAINERS}\n    session_turns: dict[int, list[tuple[dict[str, Any], str]]] = {\n        ordinal: [] for ordinal in range(SESSION_COUNT)\n    }\n    for canonical in CONTROLLED_DEMO4_V1:\n        ordinal = canonical["ordinal"]\n        session_ordinal, turn_index = _session_for_turn(ordinal)\n        outcome = _outcome_for_session(session_ordinal)\n        agents, message_content = _journey_for_turn(\n            outcome,\n            turn_index,\n            multi_turn=session_ordinal < 8,\n        )\n        agent_path = ",".join(agents)\n        handoff_count = len(agents) - 1\n        timestamp_dt = _profile_timestamp(anchor_dt, ordinal)\n        timestamp = _iso(timestamp_dt)\n        turn_epoch = int(timestamp_dt.timestamp())\n        turn_suffix = f"{ordinal:03d}"\n        session_suffix = f"{session_ordinal:03d}"\n        user_id = f"{fixture_version}-{tenant}-user-{session_suffix}"\n        session_id = f"{fixture_version}-{tenant}-session-{session_suffix}"\n        turn_id = f"{fixture_version}-{tenant}-turn-{turn_suffix}"\n        debug_id = f"{fixture_version}-{tenant}-debug-{turn_suffix}"\n        node_id = f"{fixture_version}-{tenant}-nodes-{turn_suffix}"\n        message_id = f"{fixture_version}-{tenant}-message-{turn_suffix}"\n\n        turn = {\n            "id": turn_id,\n            "type": "optimization_turn",\n            "fixture_version": fixture_version,\n            "fixture_ordinal": ordinal,\n            "session_ordinal": session_ordinal,\n            "session_outcome": outcome,\n            "tenantId": tenant,\n            "userId": user_id,\n            "sessionId": session_id,\n            "debugLogId": debug_id,\n            "complexity_tier": "default",\n            "model_deployment": PREMIUM_DEPLOYMENT,\n            "model_name": PREMIUM_MODEL,\n            "input_tokens": canonical["input_tokens"],\n            "output_tokens": canonical["output_tokens"],\n            "total_tokens": canonical["total_tokens"],\n            "cached_tokens": canonical["cached_tokens"],\n            "handoff_count": handoff_count,\n            "agent_path": agent_path,\n            "timeStamp": timestamp,\n            "turn_epoch": turn_epoch,\n        }\n        bag_values = {\n            "agent_selected": agents[-1],\n            "model_deployment": PREMIUM_DEPLOYMENT,\n            "model_name": PREMIUM_MODEL,\n            "complexity_tier": "default",\n            "model_tier": "default",\n            "input_tokens": canonical["input_tokens"],\n            "output_tokens": canonical["output_tokens"],\n            "total_tokens": canonical["total_tokens"],\n            "cached_tokens": canonical["cached_tokens"],\n            "handoff_count": handoff_count,\n            "agent_path": agent_path,\n            "tool_calls": "[]",\n            "turn_epoch": turn_epoch,\n        }\n        debug = {\n            "id": debug_id,\n            "debugLogId": debug_id,\n            "messageId": message_id,\n            "turnId": turn_id,\n            "type": "debug_log",\n            "fixture_version": fixture_version,\n            "fixture_ordinal": ordinal,\n            "session_ordinal": session_ordinal,\n            "session_outcome": outcome,\n            "tenantId": tenant,\n            "userId": user_id,\n            "sessionId": session_id,\n            "timeStamp": timestamp,\n            "turn_epoch": turn_epoch,\n            "propertyBag": [\n                {"key": key, "value": value, "timeStamp": timestamp}\n                for key, value in bag_values.items()\n            ],\n        }\n        input_parts = _split_int(canonical["input_tokens"], len(agents))\n        output_parts = _split_int(canonical["output_tokens"], len(agents))\n        cached_parts = _split_int(canonical["cached_tokens"], len(agents))\n        node_execution = {\n            "id": node_id,\n            "fixture_version": fixture_version,\n            "fixture_ordinal": ordinal,\n            "session_ordinal": session_ordinal,\n            "session_outcome": outcome,\n            "tenantId": tenant,\n            "userId": user_id,\n            "sessionId": session_id,\n            "turnId": turn_id,\n            "debugLogId": debug_id,\n            "nodeExecutions": [\n                {\n                    "seq": index,\n                    "agent": agent,\n                    "model_deployment": PREMIUM_DEPLOYMENT,\n                    "model_name": PREMIUM_MODEL,\n                    "input_tokens": input_parts[index],\n                    "output_tokens": output_parts[index],\n                    "cached_tokens": cached_parts[index],\n                    "tool_calls": 0,\n                    "recall_used": False,\n                }\n                for index, agent in enumerate(agents)\n            ],\n            "nodeCount": len(agents),\n            "timeStamp": timestamp,\n            "turn_epoch": turn_epoch,\n        }\n        message = {\n            "id": message_id,\n            "messageId": message_id,\n            "fixture_version": fixture_version,\n            "fixture_ordinal": ordinal,\n            "session_ordinal": session_ordinal,\n            "session_outcome": outcome,\n            "tenantId": tenant,\n            "userId": user_id,\n            "sessionId": session_id,\n            "turnId": turn_id,\n            "role": "assistant",\n            "content": message_content,\n            "toolCalls": [],\n            "ts": timestamp,\n            "timeStamp": timestamp,\n            "keywords": ["controlled", outcome, canonical["destination"].lower()],\n            "superseded": False,\n        }\n        manifest["OptimizationTurns"].append(turn)\n        manifest["Debug"].append(debug)\n        manifest["NodeExecutions"].append(node_execution)\n        manifest["Messages"].append(message)\n        session_turns[session_ordinal].append((turn, message_id))\n\n    for session_ordinal, linked_turns in session_turns.items():\n        first_turn = linked_turns[0][0]\n        last_turn = linked_turns[-1][0]\n        session_id = first_turn["sessionId"]\n        user_id = first_turn["userId"]\n        outcome = first_turn["session_outcome"]\n        session_suffix = f"{session_ordinal:03d}"\n        manifest["Sessions"].append({\n            "id": session_id,\n            "sessionId": session_id,\n            "fixture_version": fixture_version,\n            "fixture_ordinal": session_ordinal,\n            "session_outcome": outcome,\n            "tenantId": tenant,\n            "userId": user_id,\n            "title": f"Controlled Demo 4 journey {session_suffix}",\n            "activeAgent": "orchestrator",\n            "createdAt": first_turn["timeStamp"],\n            "lastActivityAt": last_turn["timeStamp"],\n            "status": "active",\n            "messageCount": len(linked_turns),\n        })\n        if outcome == "converted":\n            trip_id = f"{fixture_version}-{tenant}-trip-{session_suffix}"\n            manifest["Trips"].append({\n                "id": trip_id,\n                "tripId": trip_id,\n                "fixture_version": fixture_version,\n                "fixture_ordinal": session_ordinal,\n                "session_outcome": outcome,\n                "tenantId": tenant,\n                "userId": user_id,\n                "sessionId": session_id,\n                "turnId": last_turn["id"],\n                "destination": CONTROLLED_DEMO4_V1[last_turn["fixture_ordinal"]]["destination"],\n                "startDate": "2027-01-10",\n                "endDate": "2027-01-12",\n                "tripDuration": 3,\n                "status": "confirmed",\n                "days": [],\n                "createdAt": last_turn["timeStamp"],\n                "updatedAt": last_turn["timeStamp"],\n                "timeStamp": last_turn["timeStamp"],\n            })\n    return manifest\n\n\ndef allocate_burst_tiers(count: int) -> dict[str, int]:\n    """Allocate the 10/55/35 model mix with largest remainder and stable name ties."""\n    if not isinstance(count, int) or isinstance(count, bool) or count < 1 or count > TURN_COUNT:\n        raise ValueError(f"controlled burst count must be between 1 and {TURN_COUNT}")\n    exact = {\n        tier: count * profile["weight"]\n        for tier, profile in BURST_PROFILES.items()\n    }\n    allocated = {tier: int(value) for tier, value in exact.items()}\n    remaining = count - sum(allocated.values())\n    order = sorted(exact, key=lambda tier: (-(exact[tier] - allocated[tier]), tier))\n    for tier in order[:remaining]:\n        allocated[tier] += 1\n    return allocated\n\n\ndef _tier_ordinals(count: int) -> list[str]:\n    targets = allocate_burst_tiers(count)\n    used = {tier: 0 for tier in targets}\n    result: list[str] = []\n    for ordinal in range(count):\n        tier = min(\n            targets,\n            key=lambda name: (\n                -((ordinal + 1) * targets[name] / count - used[name]),\n                name,\n            ),\n        )\n        used[tier] += 1\n        result.append(tier)\n    if used != targets:\n        raise RuntimeError(f"controlled tier allocation drifted: {used} != {targets}")\n    return result\n\n\ndef _burst_timestamp(anchor: datetime, ordinal: int, count: int, window_minutes: int) -> datetime:\n    minute = (ordinal * window_minutes) // count\n    indexes = [\n        index\n        for index in range(count)\n        if (index * window_minutes) // count == minute\n    ]\n    position = indexes.index(ordinal)\n    second = (position * 60) // max(len(indexes), 1)\n    return anchor + timedelta(minutes=minute, seconds=second)\n\n\ndef _profile_timestamp(anchor: datetime, ordinal: int) -> datetime:\n    offset = ordinal\n    for minute, bucket_count in enumerate(CANONICAL_MINUTE_PROFILE):\n        if offset < bucket_count:\n            second = (offset * 60) // bucket_count\n            return anchor + timedelta(minutes=minute, seconds=second)\n        offset -= bucket_count\n    raise ValueError(f"profile ordinal must be between 0 and {sum(CANONICAL_MINUTE_PROFILE) - 1}")\n\n\ndef build_after_burst_manifest(\n    *,\n    tenant: str = "analytics",\n    fixture_version: str = FIXTURE_VERSION,\n    burst_version: str = BURST_VERSION,\n    anchor: str = BURST_ANCHOR,\n    count: int = TURN_COUNT,\n    window_minutes: int = WINDOW_MINUTES,\n) -> dict[str, list[dict[str, Any]]]:\n    """Build the deterministic Analytics after-burst from the canonical business workload."""\n    if tenant != "analytics":\n        raise ValueError("controlled traffic is supported only for tenant \'analytics\'")\n    if fixture_version != FIXTURE_VERSION:\n        raise ValueError(f"unsupported fixture version: {fixture_version}")\n    if not isinstance(burst_version, str) or not burst_version.strip():\n        raise ValueError("burst version must be non-empty")\n    if not isinstance(window_minutes, int) or isinstance(window_minutes, bool) or window_minutes < 1:\n        raise ValueError("controlled burst window must be a positive integer")\n    anchor_dt = parse_anchor(anchor)\n    tiers = _tier_ordinals(count)\n    baseline = build_tenant_manifest(tenant)  # canonical workload; identity is replaced below\n    selected_turns = baseline["OptimizationTurns"][:count]\n    selected_session_ids = {turn["sessionId"] for turn in selected_turns}\n    namespace = f"{fixture_version}:{burst_version}:{tenant}"\n\n    manifest = {name: [] for name in COHORT_CONTAINERS}\n    timestamps: dict[int, str] = {}\n    epochs: dict[int, int] = {}\n    turn_ids: dict[int, str] = {}\n    debug_ids: dict[int, str] = {}\n    message_ids: dict[int, str] = {}\n    session_ids: dict[str, str] = {}\n    user_ids: dict[str, str] = {}\n    session_ordinals = {\n        session_id: index\n        for index, session_id in enumerate(\n            dict.fromkeys(turn["sessionId"] for turn in selected_turns)\n        )\n    }\n    for old_session_id, session_ordinal in session_ordinals.items():\n        suffix = f"{session_ordinal:03d}"\n        session_ids[old_session_id] = f"{burst_version}-{tenant}-session-{suffix}"\n        user_ids[old_session_id] = f"{burst_version}-{tenant}-user-{suffix}"\n\n    for ordinal, baseline_turn in enumerate(selected_turns):\n        timestamp_dt = (\n            _profile_timestamp(anchor_dt, ordinal)\n            if count == TURN_COUNT and window_minutes == WINDOW_MINUTES\n            else _burst_timestamp(anchor_dt, ordinal, count, window_minutes)\n        )\n        timestamps[ordinal] = _iso(timestamp_dt)\n        epochs[ordinal] = int(timestamp_dt.timestamp())\n        suffix = f"{ordinal:03d}"\n        turn_ids[ordinal] = f"{burst_version}-{tenant}-turn-{suffix}"\n        debug_ids[ordinal] = f"{burst_version}-{tenant}-debug-{suffix}"\n        message_ids[ordinal] = f"{burst_version}-{tenant}-message-{suffix}"\n\n        tier = tiers[ordinal]\n        profile = BURST_PROFILES[tier]\n        old_session_id = baseline_turn["sessionId"]\n        turn = copy.deepcopy(baseline_turn)\n        turn.update({\n            "id": turn_ids[ordinal],\n            "fixture_version": fixture_version,\n            "burst_version": burst_version,\n            "controlled_namespace": namespace,\n            "burst_ordinal": ordinal,\n            "burst_anchor": anchor,\n            "burst_count": count,\n            "burst_window_minutes": window_minutes,\n            "tenantId": tenant,\n            "userId": user_ids[old_session_id],\n            "sessionId": session_ids[old_session_id],\n            "debugLogId": debug_ids[ordinal],\n            "complexity_tier": tier,\n            "model_deployment": profile["deployment"],\n            "model_name": profile["model"],\n            "timeStamp": timestamps[ordinal],\n            "turn_epoch": epochs[ordinal],\n        })\n        manifest["OptimizationTurns"].append(turn)\n\n        debug = copy.deepcopy(baseline["Debug"][ordinal])\n        debug.update({\n            "id": debug_ids[ordinal],\n            "debugLogId": debug_ids[ordinal],\n            "messageId": message_ids[ordinal],\n            "turnId": turn_ids[ordinal],\n            "fixture_version": fixture_version,\n            "burst_version": burst_version,\n            "controlled_namespace": namespace,\n            "burst_ordinal": ordinal,\n            "burst_anchor": anchor,\n            "burst_count": count,\n            "burst_window_minutes": window_minutes,\n            "userId": user_ids[old_session_id],\n            "sessionId": session_ids[old_session_id],\n            "timeStamp": timestamps[ordinal],\n            "turn_epoch": epochs[ordinal],\n        })\n        bag_updates = {\n            "model_deployment": profile["deployment"],\n            "model_name": profile["model"],\n            "complexity_tier": tier,\n            "model_tier": tier,\n            "turn_epoch": epochs[ordinal],\n        }\n        for item in debug["propertyBag"]:\n            if item["key"] in bag_updates:\n                item["value"] = bag_updates[item["key"]]\n            item["timeStamp"] = timestamps[ordinal]\n        manifest["Debug"].append(debug)\n\n        nodes = copy.deepcopy(baseline["NodeExecutions"][ordinal])\n        nodes.update({\n            "id": f"{burst_version}-{tenant}-nodes-{suffix}",\n            "turnId": turn_ids[ordinal],\n            "debugLogId": debug_ids[ordinal],\n            "fixture_version": fixture_version,\n            "burst_version": burst_version,\n            "controlled_namespace": namespace,\n            "burst_ordinal": ordinal,\n            "burst_anchor": anchor,\n            "burst_count": count,\n            "burst_window_minutes": window_minutes,\n            "userId": user_ids[old_session_id],\n            "sessionId": session_ids[old_session_id],\n            "timeStamp": timestamps[ordinal],\n            "turn_epoch": epochs[ordinal],\n        })\n        for node in nodes["nodeExecutions"]:\n            node["model_deployment"] = profile["deployment"]\n            node["model_name"] = profile["model"]\n        manifest["NodeExecutions"].append(nodes)\n\n        message = copy.deepcopy(baseline["Messages"][ordinal])\n        message.update({\n            "id": message_ids[ordinal],\n            "messageId": message_ids[ordinal],\n            "turnId": turn_ids[ordinal],\n            "fixture_version": fixture_version,\n            "burst_version": burst_version,\n            "controlled_namespace": namespace,\n            "burst_ordinal": ordinal,\n            "burst_anchor": anchor,\n            "burst_count": count,\n            "burst_window_minutes": window_minutes,\n            "userId": user_ids[old_session_id],\n            "sessionId": session_ids[old_session_id],\n            "ts": timestamps[ordinal],\n            "timeStamp": timestamps[ordinal],\n        })\n        manifest["Messages"].append(message)\n\n    turns_by_old_session: dict[str, list[int]] = {}\n    for ordinal, turn in enumerate(selected_turns):\n        turns_by_old_session.setdefault(turn["sessionId"], []).append(ordinal)\n    for baseline_session in baseline["Sessions"]:\n        old_session_id = baseline_session["sessionId"]\n        if old_session_id not in selected_session_ids:\n            continue\n        ordinals = turns_by_old_session[old_session_id]\n        session_ordinal = session_ordinals[old_session_id]\n        session = copy.deepcopy(baseline_session)\n        session.update({\n            "id": session_ids[old_session_id],\n            "sessionId": session_ids[old_session_id],\n            "fixture_version": fixture_version,\n            "burst_version": burst_version,\n            "controlled_namespace": namespace,\n            "burst_ordinal": session_ordinal,\n            "burst_anchor": anchor,\n            "burst_count": count,\n            "burst_window_minutes": window_minutes,\n            "userId": user_ids[old_session_id],\n            "createdAt": timestamps[ordinals[0]],\n            "lastActivityAt": timestamps[ordinals[-1]],\n            "messageCount": len(ordinals),\n        })\n        manifest["Sessions"].append(session)\n\n    trip_by_session = {trip["sessionId"]: trip for trip in baseline["Trips"]}\n    for old_session_id, ordinals in turns_by_old_session.items():\n        if old_session_id not in trip_by_session:\n            continue\n        session_ordinal = session_ordinals[old_session_id]\n        last_ordinal = ordinals[-1]\n        trip = copy.deepcopy(trip_by_session[old_session_id])\n        trip_id = f"{burst_version}-{tenant}-trip-{session_ordinal:03d}"\n        trip.update({\n            "id": trip_id,\n            "tripId": trip_id,\n            "fixture_version": fixture_version,\n            "burst_version": burst_version,\n            "controlled_namespace": namespace,\n            "burst_ordinal": session_ordinal,\n            "burst_anchor": anchor,\n            "burst_count": count,\n            "burst_window_minutes": window_minutes,\n            "userId": user_ids[old_session_id],\n            "sessionId": session_ids[old_session_id],\n            "turnId": turn_ids[last_ordinal],\n            "createdAt": timestamps[last_ordinal],\n            "updatedAt": timestamps[last_ordinal],\n            "timeStamp": timestamps[last_ordinal],\n        })\n        manifest["Trips"].append(trip)\n\n    for rows in manifest.values():\n        rows.sort(key=lambda item: item["id"])\n    return manifest\n\n\ndef _normalized(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {\n            key: _normalized(item)\n            for key, item in sorted(value.items())\n            if key not in _IDENTITY_FIELDS and not key.startswith("_")\n        }\n    if isinstance(value, list):\n        return [_normalized(item) for item in value]\n    return value\n\n\ndef normalized_fingerprint(manifest: dict[str, list[dict[str, Any]]]) -> str:\n    """Hash a manifest after removing tenant/document identity and Cosmos metadata."""\n    normalized = {\n        name: [_normalized(copy.deepcopy(item)) for item in items]\n        for name, items in sorted(manifest.items())\n    }\n    payload = json.dumps(normalized, sort_keys=True, separators=(",", ":"), ensure_ascii=True)\n    return hashlib.sha256(payload.encode("utf-8")).hexdigest()\n\n\ndef _invalid_evaluation(\n    tenant: str,\n    policy_status: str,\n    reason: str,\n    *,\n    baseline_expected_ids: list[str] | None = None,\n    baseline_observed_ids: list[str] | None = None,\n    burst_expected_ids: list[str] | None = None,\n    burst_observed_ids: list[str] | None = None,\n) -> dict[str, Any]:\n    return {\n        "tenant_id": tenant,\n        "policy_status": policy_status,\n        "dataset_phase": "invalid",\n        "display_state": "Invalid controlled dataset",\n        "state_valid": False,\n        "state_reason": reason,\n        "measurement_status": "invalid" if reason else "missing",\n        "fixture_version": FIXTURE_VERSION,\n        "burst_version": BURST_VERSION if tenant == "analytics" else None,\n        "burst_anchor": BURST_ANCHOR if tenant == "analytics" else None,\n        "burst_window_minutes": WINDOW_MINUTES if tenant == "analytics" else None,\n        "baseline_expected_count": len(baseline_expected_ids or []),\n        "baseline_observed_count": len(baseline_observed_ids or []),\n        "baseline_expected_ids": baseline_expected_ids or [],\n        "baseline_observed_ids": baseline_observed_ids or [],\n        "burst_expected_count": len(burst_expected_ids or []),\n        "burst_observed_count": len(burst_observed_ids or []),\n        "burst_expected_ids": burst_expected_ids or [],\n        "burst_observed_ids": burst_observed_ids or [],\n        "model_counts": {},\n        "measurement": None,\n    }\n\n\ndef _query_controlled_rows(\n    db: Any,\n    container_name: str,\n    tenant: str,\n) -> list[dict[str, Any]]:\n    container = db.get_container_client(container_name)\n    return list(container.query_items(\n        query="SELECT * FROM c WHERE c.tenantId=@tenant",\n        parameters=[{"name": "@tenant", "value": tenant}],\n        enable_cross_partition_query=True,\n    ))\n\n\ndef _raw_policy_status(db: Any, tenant: str) -> str:\n    try:\n        policy = db.get_container_client("OptimizationPolicies").read_item(\n            item=f"{tenant}::model-selection",\n            partition_key="model-selection",\n        )\n        return str(policy.get("status") or "not_proposed")\n    except Exception:  # noqa: BLE001\n        return "not_proposed"\n\n\ndef _group_expected_rows(\n    rows: list[dict[str, Any]],\n    expected_ids: set[str],\n) -> tuple[dict[str, dict[str, Any]], list[str], list[str]]:\n    grouped: dict[str, list[dict[str, Any]]] = {}\n    for row in rows:\n        row_id = row.get("id")\n        if row_id in expected_ids:\n            grouped.setdefault(str(row_id), []).append(row)\n    duplicates = sorted(row_id for row_id, values in grouped.items() if len(values) > 1)\n    conflicts = sorted(\n        row_id\n        for row_id, values in grouped.items()\n        if len(values) > 1\n        and any(\n            json.dumps(value, sort_keys=True, default=str)\n            != json.dumps(values[0], sort_keys=True, default=str)\n            for value in values[1:]\n        )\n    )\n    return {row_id: values[0] for row_id, values in grouped.items()}, duplicates, conflicts\n\n\ndef _contains_expected(actual: Any, expected: Any) -> bool:\n    if isinstance(expected, dict):\n        return isinstance(actual, dict) and all(\n            key in actual and _contains_expected(actual[key], value)\n            for key, value in expected.items()\n        )\n    if isinstance(expected, list):\n        return (\n            isinstance(actual, list)\n            and len(actual) == len(expected)\n            and all(\n                _contains_expected(actual_item, expected_item)\n                for actual_item, expected_item in zip(actual, expected)\n            )\n        )\n    return actual == expected\n\n\ndef _validate_manifest(\n    db: Any,\n    tenant: str,\n    manifest: dict[str, list[dict[str, Any]]],\n    phase: str,\n) -> tuple[dict[str, dict[str, dict[str, Any]]], str | None]:\n    observed: dict[str, dict[str, dict[str, Any]]] = {}\n    first_error: str | None = None\n    for container_name in COHORT_CONTAINERS:\n        expected = {row["id"]: row for row in manifest[container_name]}\n        rows = _query_controlled_rows(db, container_name, tenant)\n        grouped, duplicates, conflicts = _group_expected_rows(rows, set(expected))\n        observed[container_name] = grouped\n        label = f"{phase} {container_name}"\n        if conflicts:\n            first_error = first_error or (\n                f"conflicting expected {phase} IDs in {container_name}: "\n                f"{\', \'.join(conflicts)}"\n            )\n            continue\n        if duplicates:\n            first_error = first_error or (\n                f"duplicate expected {phase} IDs in {container_name}: "\n                f"{\', \'.join(duplicates)}"\n            )\n            continue\n        missing = sorted(set(expected) - set(grouped))\n        if missing:\n            first_error = first_error or (\n                f"missing expected {phase} IDs in {container_name}: "\n                f"{\', \'.join(missing)}"\n            )\n            continue\n        for row_id, actual in grouped.items():\n            expected_row = expected[row_id]\n            if container_name == "OptimizationTurns":\n                expected_row = {\n                    key: value\n                    for key, value in expected_row.items()\n                    if key not in _TURN_SPECIALIZED_FIELDS\n                }\n            if not _contains_expected(actual, expected_row):\n                first_error = first_error or (\n                    f"{label} metadata/linkage mismatch for {row_id}"\n                )\n                break\n    return observed, first_error\n\n\ndef _token_decimal(row: dict[str, Any], field: str) -> Decimal:\n    if field not in row or isinstance(row[field], bool) or not isinstance(row[field], (int, float, Decimal)):\n        raise ValueError(f"{row.get(\'id\', \'<unknown>\')} has missing/non-numeric {field}")\n    try:\n        value = Decimal(str(row[field]))\n    except (InvalidOperation, ValueError) as exc:\n        raise ValueError(f"{row.get(\'id\', \'<unknown>\')} has missing/non-numeric {field}") from exc\n    if not value.is_finite() or value < 0:\n        raise ValueError(f"{row.get(\'id\', \'<unknown>\')} has negative/non-finite {field}")\n    return value\n\n\ndef _canonical_controlled_deployment(row: dict[str, Any]) -> str:\n    deployment = row.get("model_deployment")\n    model = row.get("model_name")\n    if deployment in {PREMIUM_DEPLOYMENT, PREMIUM_MODEL} and model in {\n        PREMIUM_DEPLOYMENT,\n        PREMIUM_MODEL,\n    }:\n        return PREMIUM_DEPLOYMENT\n    exact = {\n        ("gpt-5-mini", "gpt-5-mini-2025-08-07"): "gpt-5-mini",\n        ("gpt-5-nano", "gpt-5-nano-2025-08-07"): "gpt-5-nano",\n    }\n    canonical = exact.get((deployment, model))\n    if canonical is None:\n        raise ValueError(\n            f"{row.get(\'id\', \'<unknown>\')} has unknown or conflicting deployment/model "\n            f"{deployment!r}/{model!r}"\n        )\n    return canonical\n\n\ndef _load_checked_in_pricing() -> tuple[dict[str, dict[str, Decimal]], str, str]:\n    embedded = globals().get("CONTROLLED_DEMO4_EMBEDDED_PRICING")\n    if embedded is not None:\n        raw = embedded\n        payload = json.dumps(raw, sort_keys=True, separators=(",", ":")).encode("utf-8")\n        source = str(\n            globals().get(\n                "CONTROLLED_DEMO4_EMBEDDED_PRICING_SOURCE",\n                "embedded python/data/model_pricing.json",\n            )\n        )\n        pricing_hash = str(\n            globals().get(\n                "CONTROLLED_DEMO4_EMBEDDED_PRICING_SHA256",\n                hashlib.sha256(payload).hexdigest(),\n            )\n        )\n        pricing = {\n            model: {\n                "input": Decimal(str(values["input"])),\n                "output": Decimal(str(values["output"])),\n            }\n            for model, values in raw.items()\n        }\n        return pricing, source, pricing_hash\n    path = Path(__file__).resolve().parents[3] / "data" / "model_pricing.json"\n    payload = path.read_bytes()\n    raw = json.loads(payload)\n    pricing = {\n        model: {\n            "input": Decimal(str(values["input"])),\n            "output": Decimal(str(values["output"])),\n        }\n        for model, values in raw.items()\n    }\n    return pricing, r"python\\data\\model_pricing.json", hashlib.sha256(payload).hexdigest()\n\n\ndef _build_measurement(rows: list[dict[str, Any]]) -> dict[str, Any]:\n    pricing, pricing_source, pricing_hash = _load_checked_in_pricing()\n    premium = pricing[PREMIUM_DEPLOYMENT]\n    baseline_cost = Decimal("0")\n    actual_cost = Decimal("0")\n    model_counts: Counter[str] = Counter()\n    for row in rows:\n        deployment = _canonical_controlled_deployment(row)\n        if deployment not in pricing:\n            raise ValueError(f"{row.get(\'id\', \'<unknown>\')} uses unpriced deployment {deployment}")\n        input_tokens = _token_decimal(row, "input_tokens")\n        output_tokens = _token_decimal(row, "output_tokens")\n        baseline_cost += (\n            input_tokens * premium["input"] + output_tokens * premium["output"]\n        ) / Decimal("1000000")\n        actual = pricing[deployment]\n        actual_cost += (\n            input_tokens * actual["input"] + output_tokens * actual["output"]\n        ) / Decimal("1000000")\n        model_counts[deployment] += 1\n    saving = baseline_cost - actual_cost\n    saving_pct = (\n        Decimal("100") * saving / baseline_cost\n        if baseline_cost > 0\n        else Decimal("0")\n    )\n    positive = baseline_cost > 0 and actual_cost < baseline_cost\n    return {\n        "measurement_status": "measured",\n        "measurement_kind": "Measured",\n        "measurement_scope": "Analytics controlled after burst",\n        "measurement_tenant": "analytics",\n        "fixture_version": FIXTURE_VERSION,\n        "burst_version": BURST_VERSION,\n        "burst_anchor": BURST_ANCHOR,\n        "burst_window_minutes": WINDOW_MINUTES,\n        "expected_count": TURN_COUNT,\n        "observed_count": len(rows),\n        "observed_ids": sorted(str(row["id"]) for row in rows),\n        "manifest_identity": f"{FIXTURE_VERSION}:{BURST_VERSION}:analytics",\n        "model_counts": dict(sorted(model_counts.items())),\n        "pricing_source": pricing_source,\n        "pricing_version": PRICING_VERSION,\n        "pricing_sha256": pricing_hash,\n        "validation_status": "valid",\n        "validation_reason": (\n            "complete controlled burst priced below all-premium baseline"\n            if positive\n            else "complete controlled burst has zero measured saving"\n        ),\n        "rounding_tolerance_usd": ROUNDING_TOLERANCE_USD,\n        "rounding_tolerance_pct": ROUNDING_TOLERANCE_PCT,\n        "baseline_cost_usd_unrounded": str(baseline_cost),\n        "actual_cost_usd_unrounded": str(actual_cost),\n        "saving_usd_unrounded": str(saving),\n        "saving_pct_unrounded": str(saving_pct),\n        "baseline_cost_usd": round(float(baseline_cost), 4),\n        "actual_cost_usd": round(float(actual_cost), 4),\n        "saving_usd": round(float(saving), 4),\n        "saving_pct": round(float(saving_pct), 1),\n        "positive_measured_result": positive,\n    }\n\n\ndef evaluate_controlled_demo4(\n    db: Any,\n    tenant: str,\n    *,\n    policy_status: str | None = None,\n) -> dict[str, Any]:\n    """Total source-derived state and measurement evaluator for controlled Demo 4."""\n    raw_policy_status = str(policy_status or _raw_policy_status(db, tenant))\n    if tenant not in TENANTS:\n        return _invalid_evaluation(tenant, raw_policy_status, f"unsupported controlled tenant: {tenant}")\n    try:\n        baseline_manifest = build_tenant_manifest(tenant)\n        baseline_observed, baseline_error = _validate_manifest(\n            db, tenant, baseline_manifest, "baseline"\n        )\n        baseline_rows = baseline_manifest["OptimizationTurns"]\n        baseline_expected = {row["id"]: row for row in baseline_rows}\n        observed_baseline = baseline_observed["OptimizationTurns"]\n        baseline_expected_ids = sorted(baseline_expected)\n        baseline_observed_ids = sorted(observed_baseline)\n        if baseline_error:\n            return _invalid_evaluation(\n                tenant, raw_policy_status, baseline_error,\n                baseline_expected_ids=baseline_expected_ids,\n                baseline_observed_ids=baseline_observed_ids,\n            )\n        for row_id, observed in observed_baseline.items():\n            expected = baseline_expected[row_id]\n            if (\n                observed.get("fixture_version") != FIXTURE_VERSION\n                or observed.get("fixture_ordinal") != expected["fixture_ordinal"]\n                or observed.get("complexity_tier") != "default"\n                or _canonical_controlled_deployment(observed) != PREMIUM_DEPLOYMENT\n            ):\n                return _invalid_evaluation(\n                    tenant, raw_policy_status, f"baseline metadata/model mismatch for {row_id}",\n                    baseline_expected_ids=baseline_expected_ids,\n                    baseline_observed_ids=baseline_observed_ids,\n                )\n            _token_decimal(observed, "input_tokens")\n            _token_decimal(observed, "output_tokens")\n\n        if tenant == "marvel":\n            return {\n                **_invalid_evaluation(\n                    tenant, raw_policy_status, "",\n                    baseline_expected_ids=baseline_expected_ids,\n                    baseline_observed_ids=baseline_observed_ids,\n                ),\n                "dataset_phase": "before",\n                "display_state": "Not Applied · Before",\n                "state_valid": True,\n                "state_reason": "complete all-premium Marvel baseline",\n                "measurement_status": "missing",\n            }\n\n        burst_manifest = build_after_burst_manifest()\n        burst_observed, burst_error = _validate_manifest(\n            db, tenant, burst_manifest, "burst"\n        )\n        burst_rows = burst_manifest["OptimizationTurns"]\n        burst_expected = {row["id"]: row for row in burst_rows}\n        observed_burst = burst_observed["OptimizationTurns"]\n        burst_expected_ids = sorted(burst_expected)\n        burst_observed_ids = sorted(observed_burst)\n        common = {\n            "baseline_expected_ids": baseline_expected_ids,\n            "baseline_observed_ids": baseline_observed_ids,\n            "burst_expected_ids": burst_expected_ids,\n            "burst_observed_ids": burst_observed_ids,\n        }\n        any_burst_rows = any(rows for rows in burst_observed.values())\n        if any_burst_rows and burst_error:\n            return _invalid_evaluation(\n                tenant, raw_policy_status, burst_error, **common\n            )\n        if not any_burst_rows:\n            display = (\n                "Policy Active · Awaiting Traffic · Before"\n                if raw_policy_status == "active"\n                else "Not Applied · Before"\n            )\n            result = _invalid_evaluation(tenant, raw_policy_status, "", **common)\n            result.update({\n                "dataset_phase": "before",\n                "display_state": display,\n                "state_valid": True,\n                "state_reason": "complete all-premium Analytics baseline; no controlled after burst",\n                "measurement_status": "missing",\n            })\n            return result\n        if burst_error:\n            return _invalid_evaluation(\n                tenant, raw_policy_status, burst_error, **common\n            )\n\n        model_counts: Counter[str] = Counter()\n        for row_id, observed in observed_burst.items():\n            expected = burst_expected[row_id]\n            required = {\n                "fixture_version": FIXTURE_VERSION,\n                "burst_version": BURST_VERSION,\n                "controlled_namespace": f"{FIXTURE_VERSION}:{BURST_VERSION}:analytics",\n                "burst_ordinal": expected["burst_ordinal"],\n                "burst_anchor": BURST_ANCHOR,\n                "burst_count": TURN_COUNT,\n                "burst_window_minutes": WINDOW_MINUTES,\n                "tenantId": "analytics",\n                "complexity_tier": expected["complexity_tier"],\n            }\n            if any(observed.get(key) != value for key, value in required.items()):\n                return _invalid_evaluation(\n                    tenant, raw_policy_status, f"burst metadata mismatch for {row_id}", **common\n                )\n            model_counts[_canonical_controlled_deployment(observed)] += 1\n            _token_decimal(observed, "input_tokens")\n            _token_decimal(observed, "output_tokens")\n        expected_model_counts = {\n            BURST_PROFILES[tier]["deployment"]: count\n            for tier, count in allocate_burst_tiers(TURN_COUNT).items()\n        }\n        if dict(model_counts) != expected_model_counts:\n            result = _invalid_evaluation(\n                tenant, raw_policy_status,\n                f"wrong controlled burst model mix: {dict(model_counts)} != {expected_model_counts}",\n                **common,\n            )\n            result["model_counts"] = dict(sorted(model_counts.items()))\n            return result\n        if raw_policy_status not in {"active", "reverted"}:\n            return _invalid_evaluation(\n                tenant, raw_policy_status,\n                f"complete burst conflicts with unsupported policy lifecycle {raw_policy_status!r}",\n                **common,\n            )\n        measurement = _build_measurement(list(observed_burst.values()))\n        result = _invalid_evaluation(tenant, raw_policy_status, "", **common)\n        result.update({\n            "dataset_phase": "after",\n            "display_state": (\n                "Applied · After"\n                if raw_policy_status == "active"\n                else "Policy Reverted · After Traffic Captured"\n            ),\n            "state_valid": True,\n            "state_reason": "complete deterministic Analytics after burst",\n            "measurement_status": "measured",\n            "model_counts": dict(sorted(model_counts.items())),\n            "measurement": measurement,\n        })\n        return result\n    except Exception as exc:  # noqa: BLE001\n        return _invalid_evaluation(tenant, raw_policy_status, str(exc))\n\n\ndef assert_evaluation_schema(result: dict[str, Any]) -> None:\n    """Reject evaluator drift across app, Fabric notebook, and fixture validators."""\n    if tuple(result) != EVALUATION_FIELDS:\n        raise AssertionError(\n            f"controlled Demo 4 evaluation schema drift: {tuple(result)} != {EVALUATION_FIELDS}"\n        )\n    measurement = result["measurement"]\n    if measurement is not None and tuple(measurement) != MEASUREMENT_FIELDS:\n        raise AssertionError(\n            "controlled Demo 4 measurement schema drift: "\n            f"{tuple(measurement)} != {MEASUREMENT_FIELDS}"\n        )\n', globals())

def _plain(_value):
    if hasattr(_value, "asDict"):
        return {_key: _plain(_item) for _key, _item in _value.asDict(recursive=False).items()}
    if isinstance(_value, list):
        return [_plain(_item) for _item in _value]
    if isinstance(_value, tuple):
        return [_plain(_item) for _item in _value]
    if isinstance(_value, dict):
        return {_key: _plain(_item) for _key, _item in _value.items()}
    return _value

_controlled_frames = {
    "OptimizationTurns": turns,
    "Debug": read_sql("Debug"),
    "NodeExecutions": read_sql("NodeExecutions"),
    "Sessions": read_sql("Sessions"),
    "Messages": messages,
    "Trips": trips,
}
_controlled_snapshot = {
    _name: [_plain(_row) for _row in _frame.collect()]
    for _name, _frame in _controlled_frames.items()
}
_policy_rows = [_plain(_row) for _row in policies.collect()]

class _SnapshotContainer:
    def __init__(self, _rows):
        self._rows = _rows
    def query_items(self, **_kwargs):
        return list(self._rows)
    def read_item(self, item, partition_key):
        for _row in self._rows:
            if _row.get("id") == item or _row.get("scenario") == item:
                return _row
        raise KeyError(item)

class _SnapshotDatabase:
    def get_container_client(self, _name):
        if _name == "OptimizationPolicies":
            return _SnapshotContainer(_policy_rows)
        return _SnapshotContainer(_controlled_snapshot.get(_name, []))

_controlled_state = evaluate_controlled_demo4(_SnapshotDatabase(), TENANT)
assert_evaluation_schema(_controlled_state)
_controlled_measurement = _controlled_state["measurement"]
print(
    "controlled Demo 4 source state:",
    _controlled_state["dataset_phase"],
    _controlled_state["display_state"],
    "valid=" + str(_controlled_state["state_valid"]),
    _controlled_state["state_reason"],
)
if TENANT == "analytics" and not (
    _controlled_state["state_valid"] and _controlled_state["dataset_phase"] == "after"
):
    print("MIRROR_NOT_READY: suppressing stale Applied/After and measured model-selection rows")

_state_row = (
    f"state::{TENANT}", "controlled_demo4_state", TENANT,
    _controlled_state["policy_status"], _controlled_state["dataset_phase"],
    _controlled_state["display_state"], bool(_controlled_state["state_valid"]),
    _controlled_state["state_reason"], _controlled_state["fixture_version"],
    _controlled_state["burst_version"], _controlled_state["burst_anchor"],
    _controlled_state["burst_window_minutes"],
    int(_controlled_state["baseline_expected_count"]),
    int(_controlled_state["baseline_observed_count"]),
    int(_controlled_state["burst_expected_count"]),
    int(_controlled_state["burst_observed_count"]),
    json.dumps(_controlled_state["model_counts"], sort_keys=True), now,
)
state_df = spark.createDataFrame(
    [_state_row],
    ["id", "type", "tenantId", "policy_status", "dataset_phase", "display_state",
     "state_valid", "state_reason", "fixture_version", "burst_version", "burst_anchor",
     "burst_window_minutes", "baseline_expected_count", "baseline_observed_count",
     "burst_expected_count", "burst_observed_count", "model_counts_json", "computed_at"],
)

## 2. Build the funnel (provided)
Aggregate turns into per-session stages and attach the friction signal.

In [ ]:
# ---- funnel stages per session (PROVIDED) ----
# A session is "searched" if any turn delegated (handoff>0 / agent_path hit find_places),
# "planned" if any turn's agent_path reached the itinerary step.
sess = (turns.groupBy("sessionId", "userId").agg(
    F.max(F.when((F.col("handoff_count") > 0) | F.col("agent_path").contains("find_places"), 1)
          .otherwise(0)).alias("searched"),
    F.max(F.when(F.col("agent_path").contains("itinerary"), 1).otherwise(0)).alias("planned"),
    F.sum("total_tokens").alias("tokens")))

# conversion: session-level when a Trip carries a sessionId, else user-level fallback
_conf = trips.filter(F.col("status").isin("confirmed", "completed"))
conv_sess = _conf.select("sessionId").distinct().withColumn("conv_s", F.lit(1))
conv_user = _conf.select("userId").distinct().withColumn("conv_u", F.lit(1))
sess = (sess.join(conv_sess, "sessionId", "left").join(conv_user, "userId", "left")
        .withColumn("confirmed",
                    F.when((F.col("planned") == 1) & (F.col("conv_s").isNotNull() | F.col("conv_u").isNotNull()), 1)
                     .otherwise(0)))

# friction signal from assistant Messages (feeds the cause classification below)
fr = (messages.filter(F.lower(F.col("role")) == "assistant").groupBy("sessionId").agg(
    F.max(F.when(F.lower(F.col("content")).rlike("which city|what city"), 1).otherwise(0)).alias("city_reask"),
    F.max(F.when(F.lower(F.col("content")).rlike("couldn't find|could not find|no matching|no results|nothing found"), 1)
          .otherwise(0)).alias("no_results")))
sess = sess.join(fr, "sessionId", "left").fillna(0, ["city_reask", "no_results"])
sess.groupBy("searched", "planned", "confirmed").count().show()

## 3. TODO 1 — classify the abandonment cause
This is the analytics decision you own (the notebook analog of `classify_complexity_tier`).

In [ ]:
# ---- TODO 1 (solution): classify WHY each non-converting session leaked ----
sess = sess.withColumn(
    "cause",
    F.when(F.col("confirmed") == 1, F.lit("converted"))
     .when(F.col("planned") == 1, F.lit("cart_abandon"))
     .when((F.col("searched") == 1) & (F.col("city_reask") == 1), F.lit("city_friction"))
     .when((F.col("searched") == 1) & (F.col("no_results") == 1), F.lit("no_results"))
     .when(F.col("searched") == 1, F.lit("search_stall"))
     .otherwise(F.lit("no_engagement")))
sess.groupBy("cause").count().orderBy(F.desc("count")).show()

## 4. Shape the insight rows (provided)
Flat rows, one value each, so they mirror cleanly and Power BI needs no session-math.

In [ ]:
# ---- build the flat OptimizationInsights rows (PROVIDED) ----
# One value per row so the mirror keeps every field as a column and Power BI reads it
# with trivial DAX. Types: funnel_stage, abandonment_cause, conversion_kpi.
f = sess.agg(
    F.count(F.lit(1)).alias("engaged"),
    F.sum("searched").alias("searched"),
    F.sum("planned").alias("planned"),
    F.sum("confirmed").alias("confirmed")).collect()[0]
engaged, searched, planned, confirmed = (int(f["engaged"]), int(f["searched"]),
                                         int(f["planned"]), int(f["confirmed"]))

stage_rows = [(f"funnel::{TENANT}::{s}", "funnel_stage", TENANT, s, o, v, now) for s, o, v in [
    ("engaged", 1, engaged), ("searched", 2, searched), ("planned", 3, planned), ("confirmed", 4, confirmed)]]
funnel_df = spark.createDataFrame(stage_rows, ["id", "type", "tenantId", "stage", "stage_order", "sessions", "computed_at"])

cause_pd = (sess.filter(F.col("cause") != "converted").groupBy("cause").count().collect())
cause_rows = [(f"cause::{TENANT}::{r['cause']}", "abandonment_cause", TENANT, r["cause"], int(r["count"]), now)
              for r in cause_pd]
cause_df = spark.createDataFrame(cause_rows or [("cause::none", "abandonment_cause", TENANT, "none", 0, now)],
                                 ["id", "type", "tenantId", "cause", "sessions", "computed_at"])

addressable = {r["cause"]: int(r["count"]) for r in cause_pd if r["cause"] != "no_engagement"}
biggest = max(addressable, key=addressable.get) if addressable else "none"
kpi_df = spark.createDataFrame([(
    f"kpi::{TENANT}", "conversion_kpi", TENANT, engaged, confirmed,
    round(100 * confirmed / max(engaged, 1), 1), biggest, now)],
    ["id", "type", "tenantId", "engaged", "confirmed", "conversion_rate", "biggest_leak", "computed_at"])

print("funnel:", engaged, searched, planned, confirmed, "| biggest leak:", biggest)

## 4b. Measured saving — counterfactual, keyed by optimization (provided)

A second reverse-ETL insight: the **measured** cost saving, keyed by the **optimization**
(scenario) — not the tenant. We price every captured turn (all tenants) under the model it
actually ran on vs. the single premium baseline (gpt-5.1), so the gap **is** the realized
saving from capability-tiered model selection. We also aggregate the mirrored **ApiEvents**
recall telemetry into the **memory-retention** saving (input tokens recalls avoided by
dropping pruned memories — `$0` until that policy is applied). These flat `optimization_result`
rows (stored under a reserved `_global_optimizations` partition key — a non-tenant bucket) feed
the report's **Measured Saving** page, where a `scenario` slicer switches between optimizations.
Run this cell, then include `result_df` in the reverse-ETL write below.

In [ ]:
# ---- 4b. Measured saving: counterfactual, keyed by OPTIMIZATION (PROVIDED) ----
# Load shared pricing, then calculate the current tenant's saving for tenant-scoped analyst cards.
# The controller's Analytics activity separately computes the all-tenant result rows below.
BASELINE_DEPLOYMENT = "gpt-5.1"

_pricing = (spark.read.format("jdbc")
            .option("url", _jdbc)
            .option("dbtable", f"[{SOURCE_SCHEMA}].[Configuration]")
            .option("accessToken", _sql_token)
            .load()
            .where(F.col("type") == "model_pricing")
            .select(F.col("model").alias("dep"),
                    F.col("input_price").cast("double").alias("in_price"),
                    F.col("output_price").cast("double").alias("out_price")))

_base = _pricing.where(F.col("dep") == BASELINE_DEPLOYMENT).collect()
b_in = float(_base[0]["in_price"]) if _base else 1.25
b_out = float(_base[0]["out_price"]) if _base else 10.0

# Tenant pricing stays per-run because the tenant-scoped analyst and metrics need it.
_tenant_priced = (turns.join(_pricing, turns["model_deployment"] == _pricing["dep"], "left")
                  .withColumn("in_price", F.coalesce(F.col("in_price"), F.lit(b_in)))
                  .withColumn("out_price", F.coalesce(F.col("out_price"), F.lit(b_out))))
_tenant_agg = _tenant_priced.agg(
    F.sum((F.col("input_tokens") * F.col("in_price") + F.col("output_tokens") * F.col("out_price")) / F.lit(1e6)).alias("actual"),
    F.sum((F.col("input_tokens") * F.lit(b_in) + F.col("output_tokens") * F.lit(b_out)) / F.lit(1e6)).alias("baseline"),
    F.count(F.lit(1)).alias("turns")).collect()[0]
_actual = float(_tenant_agg["actual"] or 0.0)
_baseline = float(_tenant_agg["baseline"] or 0.0)
_n = int(_tenant_agg["turns"] or 0)
_saving = _baseline - _actual
_saving_pct = round(100 * _saving / _baseline, 1) if _baseline else 0.0

# Global measured-saving work runs once, in the analytics controller activity.
if not RUN_GLOBAL_INSIGHTS:
    result_df = None
    print(f"Skipping global measured-saving analysis for {TENANT}; the analytics activity owns it.")
else:
    # The model-selection result comes only from the complete controlled
    # Analytics after burst evaluated above. Completeness and measured values therefore come from
    # the same captured DirectQuery snapshot; stale mirror rows cannot publish Applied/After.
    _measured_ready = (
        _controlled_state["state_valid"]
        and _controlled_state["dataset_phase"] == "after"
        and _controlled_measurement is not None
        and _controlled_measurement["positive_measured_result"]
    )
    _global_actual = float(_controlled_measurement["actual_cost_usd"]) if _measured_ready else 0.0
    _global_baseline = float(_controlled_measurement["baseline_cost_usd"]) if _measured_ready else 0.0
    _global_n = int(_controlled_measurement["observed_count"]) if _measured_ready else 0
    _global_saving = float(_controlled_measurement["saving_usd"]) if _measured_ready else 0.0
    _global_saving_pct = float(_controlled_measurement["saving_pct"]) if _measured_ready else 0.0

    # memory-retention: MEASURED from recall telemetry over the mirrored ApiEvents. Each
    # `recall_pruned_avoided` event's response.avoided_input_tokens = the input tokens a recall
    # avoided by dropping a pruned (superseded) memory from its top-k. Priced at the baseline
    # input rate. Reads $0 until the memory-retention policy is applied and recalls run - an
    # honest telemetry measurement, never an estimate. ApiEvents is high-volume, so we offload
    # its aggregation to Fabric here (not the app plane).
    import json as _json_mr
    _mr_recalls = 0
    _mr_avoided = 0
    try:
        _apiev_df = (spark.read.format("jdbc").option("url", _jdbc)
                     .option("dbtable", f"[{SOURCE_SCHEMA}].[ApiEvents]")
                     .option("accessToken", _sql_token).load())
        # An EMPTY mirrored container surfaces only system columns (_rid/_ts) - no data schema
        # yet - so guard on the columns existing before filtering (a raw UNRESOLVED_COLUMN error
        # would look like a failure). ApiEvents fills at runtime (the Module 08 memory demo), so
        # $0 here is expected, not an error.
        if all(_c in _apiev_df.columns for _c in ("provider", "operation", "response")):
            _apiev = (_apiev_df
                      .where((F.col("provider") == "memory") & (F.col("operation") == "recall_pruned_avoided"))
                      .select("response").collect())
            _mr_recalls = len(_apiev)
            for _e in _apiev:
                _resp = _e["response"]
                if isinstance(_resp, str):
                    try:
                        _resp = _json_mr.loads(_resp)
                    except Exception:
                        _resp = {}
                _mr_avoided += int((_resp or {}).get("avoided_input_tokens") or 0)
        else:
            print("memory-retention: ApiEvents has no telemetry rows yet (empty mirror) - reads $0 until the Module 08 memory demo runs (expected, not an error).")
    except Exception as _mrx:
        print("memory-retention telemetry skipped:", _mrx)
    _mr_saving = round(_mr_avoided * b_in / 1e6, 4)
    _MR_NOTE = ("Measured from recall telemetry (ApiEvents) - input tokens avoided by dropping pruned "
                "memories from a recall's top-k. Reads $0 until the memory-retention policy is applied and recalls run.")

    # One row per applyable optimization so the report's `scenario` slicer switches between them.
    # model-selection carries the real counterfactual; memory-retention the recall-telemetry
    # measurement (both computed here in Fabric); tool-call-dedup is a GOVERNED-path fix (a
    # prompt/code PR, not an in-app policy) - no measured before/after. compute_insights.py emits
    # the identical rows app-plane (idempotent by id), so notebook == console == report.
    _GOVERNED_NOTE = "Governed-path fix (human-reviewed prompt/code PR) - no in-app policy to apply, so no measured before/after here; see the turn-grain estimate on Discovered Opportunities."
    _model_result_type = "optimization_result" if _measured_ready else "suppressed_result"
    _model_note = (
        "Measured — Analytics controlled burst " + str(_controlled_state["burst_version"])
        if _measured_ready
        else "Suppressed — controlled Analytics source is incomplete, corrupt, or delayed: "
             + str(_controlled_state["state_reason"])
    )
    _result_rows = [
        ("result::model-selection", _model_result_type, "_global_optimizations",
         "model-selection", "Capability-tiered model selection", "measured",
         _global_n, round(_global_baseline, 4), round(_global_actual, 4),
         round(_global_saving, 4), _global_saving_pct, _model_note, now,
         "Measured", "Analytics controlled after burst", "analytics",
         _controlled_state["burst_version"],
         _controlled_state["policy_status"], _controlled_state["dataset_phase"],
         _controlled_state["display_state"], bool(_controlled_state["state_valid"]),
         _controlled_state["state_reason"]),
        ("result::memory-retention", "optimization_result", "_global_optimizations",
         "memory-retention", "Memory retention (prune superseded)", "telemetry",
         _mr_recalls, 0.0, 0.0, _mr_saving, 0.0, _MR_NOTE, now,
         "Measured", "Global memory recall telemetry", "", "", "", "", "", True, ""),
    ]
    for _sc, _title in [("tool-call-dedup", "Redundant tool-call dedup")]:
        _result_rows.append((f"result::{_sc}", "optimization_result", "_global_optimizations",
                             _sc, _title, "governed", 0, 0.0, 0.0, 0.0, 0.0, _GOVERNED_NOTE, now,
                             "Projected", "Analytics recommendation", "analytics",
                             _controlled_state["burst_version"],
                             _controlled_state["policy_status"], _controlled_state["dataset_phase"],
                             _controlled_state["display_state"], bool(_controlled_state["state_valid"]),
                             _controlled_state["state_reason"]))

    result_df = spark.createDataFrame(
        _result_rows,
        ["id", "type", "tenantId", "scenario", "title", "method", "turns",
         "baseline_cost_usd", "actual_cost_usd", "saving_usd", "saving_pct", "note", "computed_at",
         "measurement_kind", "measurement_scope", "measurement_tenant", "burst_version", "policy_status",
         "dataset_phase", "display_state", "state_valid", "state_reason"])
    print("measured saving: model-selection $%.4f (%.1f%% vs baseline) over %d turns; memory-retention $%.4f (%d recalls); +1 governed" % (_global_saving, _global_saving_pct, _global_n, _mr_saving, _mr_recalls))

## 5. TODO 2 — reverse-ETL the insights back to Cosmos
The pattern that closes the loop: land Fabric-computed intelligence in the operational store.

In [ ]:
# ---- TODO 2 (solution): reverse-ETL write back to Cosmos ----
cosmos_write = {
    "spark.cosmos.accountEndpoint": COSMOS_ENDPOINT,
    "spark.cosmos.account.tenantId": TENANT_ID,
    "spark.cosmos.accountDataResolverServiceName": "com.azure.cosmos.spark.fabric.FabricAccountDataResolver",
    "spark.cosmos.auth.type": "AccessToken",
    "spark.cosmos.useGatewayMode": "true",
    "spark.cosmos.database": COSMOS_DATABASE,
    "spark.cosmos.container": INSIGHTS_CONTAINER,
    "spark.cosmos.write.strategy": "ItemOverwrite",
    "spark.cosmos.write.bulk.enabled": "true",
}
tenant_frames = (funnel_df, cause_df, kpi_df, state_df)
global_frames = (result_df,) if RUN_GLOBAL_INSIGHTS else ()
for df in tenant_frames + global_frames:
    df.write.format("cosmos.oltp").options(**cosmos_write).mode("append").save()
print(f"Reverse-ETL complete for {TENANT}; global rows written={RUN_GLOBAL_INSIGHTS}.")

In [ ]:
# Persist the last completed stage because failed Fabric jobs do not retain cell output.
def _checkpoint(stage):
    status_df = spark.createDataFrame(
        [(f"run-status::{TENANT}", "notebook_run_status", TENANT, stage,
          datetime.now(timezone.utc).isoformat())],
        ["id", "type", "tenantId", "last_completed_stage", "updated_at"],
    )
    status_df.write.format("cosmos.oltp").options(**cosmos_write).mode("append").save()
    print(f"Notebook checkpoint: {stage}")

_checkpoint("core_reverse_etl")

## 5b. Agent-path cost concentration (provided)
Where do the tokens actually go? A few **agent paths** (typically the itinerary path) dominate token cost — many times a plain supervisor turn. This provided section aggregates the tenant's turns by `agent_path` and reverse-ETLs the **top paths by average tokens** as `agent_path_cost` rows (the twin of `compute_insights.py`). The Power BI **Agent Collaboration / Agent-Path Cost** page reads them — it's where tiering and tool-dedup fixes pay off.

In [ ]:
# ---- 5b. agent-path cost concentration -> reverse-ETL agent_path_cost (PROVIDED) ----
# Cost concentrated in a few agent_paths. Top 6 by avg tokens (matches compute_insights.py's
# build_agent_path_diagnostic), so the report reads the same shape whichever producer ran.
_ap = (turns.withColumn("agent_path", F.coalesce(F.col("agent_path"), F.lit("unknown")))
       .groupBy("agent_path")
       .agg(F.count(F.lit(1)).alias("turns"), F.sum("total_tokens").alias("total_tokens"))
       .withColumn("avg_tokens", F.round(F.col("total_tokens") / F.col("turns")).cast("long"))
       .orderBy(F.desc("avg_tokens")).limit(6).collect())
agentpath_rows = [(f"path::{TENANT}::{i}", "agent_path_cost", TENANT, r["agent_path"],
                   int(r["turns"]), int(r["total_tokens"] or 0), int(r["avg_tokens"] or 0), now)
                  for i, r in enumerate(_ap)]
agentpath_df = spark.createDataFrame(
    agentpath_rows or [(f"path::{TENANT}::none", "agent_path_cost", TENANT, "none", 0, 0, 0, now)],
    ["id", "type", "tenantId", "agent_path", "turns", "total_tokens", "avg_tokens", "computed_at"])
agentpath_df.write.format("cosmos.oltp").options(**cosmos_write).mode("append").save()
print(f"Agent-path reverse-ETL complete -> {len(agentpath_rows)} paths")

In [ ]:
_checkpoint("agent_path")

## 5c. Turn metrics (provided)
The aggregate KPIs the **Optimization Console** displays — total turns/tokens, estimated cost, trivial-turn share, model distribution, and a cost-by-tier breakdown. Reverse-ETL'ing them (one `turn_metrics` row with a nested `metrics` object) means the Console reads a pre-computed result instead of re-aggregating Cosmos on every request. Same shape as `compute_insights.py`'s `build_turn_metrics`.

In [ ]:
# ---- 5c. turn metrics -> reverse-ETL turn_metrics (PROVIDED) ----
# The Console KPIs, computed once here. Priced with the same mirrored Configuration pricing
# as Section 4b. Nested `metrics` object -> we build it in the driver and write it with an
# EXPLICIT schema (the Cosmos connector needs typed nesting; heterogeneous inference fails).
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               DoubleType, MapType, ArrayType)

_tp = (turns.join(_pricing, turns["model_deployment"] == _pricing["dep"], "left")
       .withColumn("in_price", F.coalesce(F.col("in_price"), F.lit(b_in)))
       .withColumn("out_price", F.coalesce(F.col("out_price"), F.lit(b_out)))
       .withColumn("cost", (F.col("input_tokens") * F.col("in_price")
                            + F.col("output_tokens") * F.col("out_price")) / F.lit(1e6)))
_m = _tp.agg(
    F.count(F.lit(1)).alias("total_turns"),
    F.sum("input_tokens").alias("in"), F.sum("output_tokens").alias("out"),
    F.sum("total_tokens").alias("tok"), F.sum("cost").alias("cost"),
    F.sum(F.when(F.col("output_tokens") < 60, 1).otherwise(0)).alias("trivial")).collect()[0]
_total = int(_m["total_turns"] or 0)
_dist = {str(r["model_name"]): int(r["count"]) for r in turns.groupBy("model_name").count().collect()}
_tier_rows = (_tp.groupBy("complexity_tier", "model_deployment")
              .agg(F.count(F.lit(1)).alias("turns"), F.sum("total_tokens").alias("tokens"),
                   F.sum("cost").alias("cost")).collect())
_by_tier = [{"complexity_tier": r["complexity_tier"] or "default",
             "deployment": r["model_deployment"] or "Unknown",
             "turns": int(r["turns"]), "tokens": int(r["tokens"] or 0),
             "cost": round(float(r["cost"] or 0), 6)}
            for r in sorted(_tier_rows, key=lambda x: -(x["cost"] or 0))]
_est_cost = round(float(_m["cost"] or 0), 4)
_metrics = {
    "tenant_id": TENANT, "total_turns": _total,
    "total_input_tokens": int(_m["in"] or 0), "total_output_tokens": int(_m["out"] or 0),
    "total_tokens": int(_m["tok"] or 0), "estimated_cost_usd": _est_cost,
    "trivial_turns": int(_m["trivial"] or 0),
    "trivial_pct": round(100 * int(_m["trivial"] or 0) / max(_total, 1), 1),
    "distinct_models": len(_dist), "model_distribution": _dist,
    "confirmed_outcomes": int(confirmed),
    "cost_per_outcome_usd": round(_est_cost / confirmed, 4) if confirmed else None,
    "by_tier": _by_tier,
}
_metrics_schema = StructType([
    StructField("id", StringType()), StructField("type", StringType()),
    StructField("tenantId", StringType()),
    StructField("metrics", StructType([
        StructField("tenant_id", StringType()), StructField("total_turns", LongType()),
        StructField("total_input_tokens", LongType()), StructField("total_output_tokens", LongType()),
        StructField("total_tokens", LongType()), StructField("estimated_cost_usd", DoubleType()),
        StructField("trivial_turns", LongType()), StructField("trivial_pct", DoubleType()),
        StructField("distinct_models", LongType()),
        StructField("model_distribution", MapType(StringType(), LongType())),
        StructField("confirmed_outcomes", LongType()), StructField("cost_per_outcome_usd", DoubleType()),
        StructField("by_tier", ArrayType(StructType([
            StructField("complexity_tier", StringType()), StructField("deployment", StringType()),
            StructField("turns", LongType()), StructField("tokens", LongType()),
            StructField("cost", DoubleType())]))),
    ])),
    StructField("computed_at", StringType()),
])
metrics_df = spark.createDataFrame(
    [(f"metrics::{TENANT}", "turn_metrics", TENANT, _metrics, now)], _metrics_schema)
metrics_df.write.format("cosmos.oltp").options(**cosmos_write).mode("append").save()
print(f"Turn-metrics reverse-ETL complete -> {_total} turns, est ${_est_cost}, {len(_by_tier)} tiers")

In [ ]:
_checkpoint("turn_metrics")

## 5d. Agent scorecard — agent × dimension health from node-grain (provided)
The primary ADR-0010 surface: **how is each individual agent doing?** Reads the mirrored **`NodeExecutions`** (per-agent node-grain), flattens each turn's executions, and scores every agent across the three node-grain dimensions (`cost_efficiency`, `model_selection`, `workflow_efficiency`) — mirroring `src/app/engine/scorecard` (rollup + dimensions), validated to match `build_scorecard` exactly. Reverse-ETL'd as `agent_scorecard` rows (one per agent×dimension) that the Power BI **Agent Performance** page (6b) reads. `compute_insights.py` writes the identical rows app-plane, so notebook == console == report.

In [ ]:
# ---- 5d. agent scorecard -> reverse-ETL agent_scorecard (over mirrored NodeExecutions) ----
# Mirrors engine/scorecard/{rollup,dimensions}.py. Uses the existing Configuration pricing
# (_pricing, loaded in 4b - per 1M tokens) so notebook cost == Console cost (the console/
# compute_insights pass the same pricing to build_scorecard). Shares/statuses are price-invariant.
# Node-grain is small, so flatten the per-turn arrays on the driver and score in Python.
import json as _json_ne
_SC_PRICE = {r['dep']: (float(r['in_price']), float(r['out_price'])) for r in _pricing.collect()}  # Configuration, per 1M
_SC_LOW_OUT = 250; _SC_PREMIUM = {'gpt-5.1', 'gpt-5'}; _SC_CHEAP = 'gpt-5-mini'
_SC_ORDER = {'opportunity': 0, 'watch': 1, 'ok': 2, 'n/a': 3}
def _sc_tcost(dep, i, o):
    p = _SC_PRICE.get(dep)
    return ((i * p[0] + o * p[1]) / 1e6) if p else 0.0

# nodeExecutions arrives as a JSON string over the SQL endpoint; tolerate an already-parsed list too.
_ne_rows = read_sql('NodeExecutions').select('turnId', 'nodeExecutions').collect()
_nodes = []
for _r in _ne_rows:
    _ne = _r['nodeExecutions']
    if isinstance(_ne, str):
        try:
            _ne = _json_ne.loads(_ne)
        except Exception:
            _ne = []
    for _nd in (_ne or []):
        _d = _nd.asDict() if hasattr(_nd, 'asDict') else dict(_nd)
        _nodes.append({'turn_id': _r['turnId'], 'agent': _d.get('agent', ''),
                       'model_deployment': _d.get('model_deployment') or _d.get('model_name') or 'Unknown',
                       'input_tokens': int(_d.get('input_tokens') or 0), 'output_tokens': int(_d.get('output_tokens') or 0)})

_by_agent = {}
for _nd in _nodes:
    _by_agent.setdefault(_nd['agent'], []).append(_nd)
_sc_total = sum(_sc_tcost(n['model_deployment'], n['input_tokens'], n['output_tokens']) for n in _nodes) or 0.0
_scorecard_rows = []
for _ag, _an in _by_agent.items():
    _cost = sum(_sc_tcost(n['model_deployment'], n['input_tokens'], n['output_tokens']) for n in _an)
    _share = (_cost / _sc_total) if _sc_total else 0.0
    _ce = 'watch' if _share >= 0.5 else 'ok'
    _execs = len(_an) or 1
    _cand = [n for n in _an if n['model_deployment'] in _SC_PREMIUM and n['output_tokens'] < _SC_LOW_OUT]
    _sav = sum(max(0.0, _sc_tcost(n['model_deployment'], n['input_tokens'], n['output_tokens']) - _sc_tcost(_SC_CHEAP, n['input_tokens'], n['output_tokens'])) for n in _cand)
    _ms = 'opportunity' if (len(_cand) / _execs >= 0.2 and _sav > 0) else 'ok'
    _per = {}
    for n in _an:
        _per[n['turn_id']] = _per.get(n['turn_id'], 0) + 1
    _turns = len(_per) or 1
    _rept = sum(1 for c in _per.values() if c > 1)
    _we = 'opportunity' if (_rept / _turns) >= 0.1 else 'ok'
    _tokens = sum(n['input_tokens'] + n['output_tokens'] for n in _an)
    _tpt = round(_tokens / _turns, 1)
    _astat = min((_ce, _ms, _we), key=lambda s: _SC_ORDER.get(s, 9))
    _hce = '$%.4f (%.0f%% of turn spend), %.0f tok/turn' % (_cost, _share * 100, _tokens / _turns)
    _hms = ('%d/%d trivial turns on a premium model -> save $%.4f by routing to %s' % (len(_cand), _execs, _sav, _SC_CHEAP)) if _cand else ('%d exec(s), no premium-on-trivial waste' % _execs)
    _hwe = ('repeats within a turn in %d/%d turns (%.0f%%)' % (_rept, _turns, _rept / _turns * 100)) if _rept else ('one call per turn across %d turns (no redundant hops)' % _turns)
    _dims = {'cost_efficiency': (_ce, _hce, round(_cost, 6), '$/window'),
             'model_selection': (_ms, _hms, round(_sav, 6), '$/window'),
             'workflow_efficiency': (_we, _hwe, round(_rept / _turns, 4), 'repeat-turn rate')}
    for _dim, (_dstat, _head, _val, _unit) in _dims.items():
        _scorecard_rows.append(('scorecard::%s::%s::%s' % (TENANT, _ag, _dim), 'agent_scorecard', TENANT, _ag, _astat,
                                round(_cost, 6), round(_share, 4), int(_execs), int(_turns), float(_tpt), _dim, _dstat, _head, float(_val), _unit, now))

_sc_cols = ['id', 'type', 'tenantId', 'agent', 'agent_status', 'cost', 'cost_share', 'executions', 'turns', 'tokens_per_turn', 'dimension', 'dim_status', 'headline', 'value', 'unit', 'computed_at']
_sc_df = spark.createDataFrame(
    _scorecard_rows or [('scorecard::%s::none' % TENANT, 'agent_scorecard', TENANT, 'none', 'n/a', 0.0, 0.0, 0, 0, 0.0, 'none', 'n/a', '', 0.0, '', now)],
    _sc_cols)
_sc_df.write.format('cosmos.oltp').options(**cosmos_write).mode('append').save()
print('Agent scorecard reverse-ETL complete -> %d rows (%d agents)' % (len(_scorecard_rows), len(_by_agent)))

In [ ]:
_checkpoint("agent_scorecard")

## 6. Memory intelligence (provided) — reverse-ETL memory health
Memories aren't free: every recall retrieves and *pays* (tokens + latency) for what it pulls, so **stale, low-salience, and superseded** memories are cost with no benefit. This provided section reads the mirrored **`memories`** table (the same SQL-endpoint path as above), computes salience / health / supersession, and reverse-ETLs the result to `OptimizationInsights` — the funnel pattern, for the **memory pillar**.

> **Reserved partition key, not a tenant.** `OptimizationInsights` is partitioned by `/tenantId`, and a *tenant* here is a customer with its own users (e.g. `marvel`, `analytics`). Memory is **global** — memories are keyed by user, not tenant — so these rows use a reserved partition key **`_global_memory`** (a bucket for non-tenant rows, distinguished by `type`), never a real tenant. The Power BI **Memory Intelligence** page reads them by `type`.

In [ ]:
# Global memory is intentionally computed only by the controller's first activity.
if not RUN_GLOBAL_INSIGHTS:
    print(f"Skipping global memory analysis for {TENANT}; the analytics activity owns it.")
else:
    # ---- memory intelligence: read mirrored `memories`, compute health, reverse-ETL (PROVIDED) ----
    # `_global_memory` is a RESERVED partition key, NOT a tenant. OptimizationInsights is
    # partitioned by /tenantId; a tenant is a customer with users (marvel, analytics). Memory is
    # global (memories are keyed by user_id/thread_id), so its rows use this reserved bucket and are
    # distinguished by `type` — they never mix with real per-tenant rows.
    MEMORY_PARTITION = "_global_memory"

    # memories are keyed by user_id/thread_id (NOT tenant) -> read the whole mirrored table via the
    # same SQL-endpoint JDBC path as read_sql, minus the tenant filter.
    mem = (spark.read.format("jdbc")
           .option("url", _jdbc)
           .option("dbtable", f"[{SOURCE_SCHEMA}].[memories]")
           .option("accessToken", _sql_token).load())
    if "embedding" in mem.columns:
        mem = mem.drop("embedding")       # skip the large vector column

    # Salience tier thresholds come from the mirrored Configuration table (type="memory_config") —
    # the single source of truth shared with compute_insights.py, so the tiers never drift. Falls
    # back to the built-in defaults if the row isn't seeded.
    _mc = (spark.read.format("jdbc")
           .option("url", _jdbc)
           .option("dbtable", f"[{SOURCE_SCHEMA}].[Configuration]")
           .option("accessToken", _sql_token).load()
           .where(F.col("type") == "memory_config").collect())
    SAL_HIGH = float(_mc[0]["salience_high"]) if _mc else 0.8
    SAL_MED = float(_mc[0]["salience_medium"]) if _mc else 0.5
    HIGH_L, MED_L, LOW_L = f"High (>={SAL_HIGH})", f"Medium ({SAL_MED}-{SAL_HIGH})", f"Low (<{SAL_MED})"
    # Some memory types (e.g. procedural) carry no salience score. NULL salience is its own
    # "Unscored" tier in BOTH breakdowns below — never folded into "Low"/"Low-value", so the
    # salience and health views stay consistent and unscored memories aren't mistaken for weak ones.
    UNSCORED_L = "Unscored"

    # supersession is marked by the 'superseded_by' pointer (+ superseded_at / supersede_reason),
    # populated only once conflict resolution has replaced a memory
    _sup = F.col("superseded_by").isNotNull() if "superseded_by" in mem.columns else F.lit(False)
    mem = (mem
           .withColumn("salience_tier",
                       F.when(F.col("salience").isNull(), UNSCORED_L)
                        .when(F.col("salience") >= SAL_HIGH, HIGH_L)
                        .when(F.col("salience") >= SAL_MED, MED_L)
                        .otherwise(LOW_L))
           .withColumn("memory_health",
                       F.when(_sup == True, "Superseded")
                        .when(F.col("salience").isNull(), UNSCORED_L)
                        .when(F.col("salience") < SAL_MED, "Low-value")
                        .otherwise("Active")))

    total = mem.count()
    # Salience KPIs are computed over SCORED memories only (salience IS NOT NULL). Unscored
    # memories (e.g. procedural guidance rules) have no strength score, so folding them into
    # these denominators would understate avg salience and skew the health rates. total_memories
    # still counts everything; scored_memories exposes the salience denominator explicitly.
    a = mem.agg(F.avg("salience").alias("avg"),
                F.sum(F.when(F.col("salience").isNotNull(), 1).otherwise(0)).alias("scored"),
                F.sum(F.when(_sup == True, 1).otherwise(0)).alias("sup"),
                F.sum(F.when(F.col("salience") < SAL_MED, 1).otherwise(0)).alias("low")).collect()[0]
    scored = int(a["scored"] or 0)
    avg_sal = round(float(a["avg"] or 0), 3)
    sup_pct = round(100 * int(a["sup"] or 0) / max(total, 1), 1)
    low_pct = round(100 * int(a["low"] or 0) / max(scored, 1), 1)

    mem_kpi_df = spark.createDataFrame(
        [(f"memkpi::{MEMORY_PARTITION}", "memory_kpi", MEMORY_PARTITION, total, scored, avg_sal, sup_pct, low_pct, now)],
        ["id", "type", "tenantId", "total_memories", "scored_memories", "avg_salience", "supersession_rate", "low_salience_rate", "computed_at"])

    def _mem_buckets(col, rowtype):
        rs = mem.groupBy(col).count().collect()
        data = [(f"{rowtype}::{MEMORY_PARTITION}::{r[col]}", rowtype, MEMORY_PARTITION, str(r[col]), int(r["count"]), now)
                for r in rs]
        return spark.createDataFrame(
            data or [(f"{rowtype}::none", rowtype, MEMORY_PARTITION, "none", 0, now)],
            ["id", "type", "tenantId", "label", "count", "computed_at"])

    for df in (mem_kpi_df,
               _mem_buckets("type", "memory_type"),
               _mem_buckets("salience_tier", "memory_salience"),
               _mem_buckets("memory_health", "memory_health")):
        df.write.format("cosmos.oltp").options(**cosmos_write).mode("append").save()
    print(f"Memory reverse-ETL complete -> {total} memories ({scored} scored), avg salience {avg_sal}, {sup_pct}% superseded, {low_pct}% low-salience (of scored)")

In [ ]:
_checkpoint("memory_intelligence")

## 7. LLM analyst — the model *proposes*, the engine *disposes* (provided)

The final, highest-maturity step of the loop (ADR-0010 Layer 2): turn the aggregate
telemetry above into **ranked recommendations** with an **LLM analyst**, safely.

The model **proposes** one optimization card as strict JSON **per detected opportunity**
— here two: the capability-tiered **model-selection** downgrade-candidate estimate (seam `config`) and
the **repeated-node / tool-call-dedup** structural finding (seam `prompt`,
`supervisor.prompty`). Five deterministic **guardrails** then **dispose** each card:
1. **bounded** — the card's seam/target must be on the app's *declared* surface (else reject);
2. **cited** — every card must cite the detector + opportunity id (else reject);
3. **engine computes the saving** — the model's dollar figure is **ignored**; the
   engine estimate wins (tenant premium-short-output candidates re-priced from gpt-5.1
   to gpt-5-nano for model-selection; the priced
   avoidable duplicate hop for tool-dedup);
4. **apply_mode from the seam** — `config` auto-applies; `prompt`/`code` are staged;
5. **autonomy ceiling from the seam** — `config` L4, `prompt`/`code` L3.

Only detections with positive count **and** positive saving reach the analyst. We reverse-ETL
each accepted card two ways: a `discovered_opportunity` row (the
analyst's native output) **and** a flat `recommendation_card` projection the Power BI
**Discovered Opportunities** page and the Console already read — so the tool-dedup card,
once analyzed, **supersedes** the app-plane "insight (awaiting analysis)" card on the same
`tool-call-dedup` id. When a previously positive signal disappears, the same deterministic
discovery, agent-opportunity, and recommendation IDs are overwritten as
`suppressed_opportunity` tombstones. The call is **keyless** (Entra token to the app's Azure OpenAI); if
`AOAI_ENDPOINT` is blank or the call fails, the analyst falls back to the deterministic
proposer, so the section always lands guardrailed cards. This mirrors the reusable engine
analyst in `src/app/engine/analyst/llm.py` and `pipeline.analyze` — same prompt, same
parse, same guardrails, one card per detection.

In [ ]:
# ---- 7. LLM analyst: propose -> guardrail -> reverse-ETL (PROVIDED) ----
# Mirrors src/app/engine/analyst/llm.py + guardrails.py (single design, two runtimes).
import json as _json
import requests as _rq

# The declared optimizable surface — kept in lockstep with src/app/engine/seams/catalog.py.
# Guardrail #1 bounds the analyst to exactly these seams/targets.
SURFACE = {"config": ["model-selection"],
           "prompt": ["itinerary_agent.prompty", "supervisor.prompty"],
           "code": ["introduce-model-selector"]}
SEAM_APPLY_MODE = {"config": "auto", "prompt": "staged_change", "code": "staged_change"}
SEAM_CEILING = {"config": "L4", "prompt": "L3", "code": "L3"}
# Which seam/target each opportunity is fixed at — mirrors pipeline.OPPORTUNITY_SEAMS.
OPPORTUNITY_SEAMS = {"opp-modelfit-supervisor": ("config", "model-selection"),
                     "opp-repeated-node": ("prompt", "supervisor.prompty")}

SYSTEM = (
    "You are an optimization analyst for a multi-agent app. Given a detected issue, propose "
    "exactly ONE change as STRICT JSON (no prose, no markdown) with keys:\n"
    '  seam: one of "config" | "prompt" | "code"\n'
    "  target: MUST be one of the allowed targets for that seam (given below)\n"
    "  claimed_saving: number (your best dollar estimate)\n"
    '  apply_mode: "auto" or "staged_change"\n'
    '  autonomy_ceiling: "L3" | "L4" | "L5"\n'
    "  evidence: a list with one object {detector, opportunity_id, traces:[...]}\n"
    "Cite the detector + opportunity id you were given. Output ONLY the JSON object."
)

# ---- the ENGINE-computed saving for the repeated-node (tool-dedup) opportunity ----
# Turn-grain estimate mirroring engine/projection/tool_dedup.py: a turn whose agent_path
# repeats the same non-supervisor agent back-to-back wastes ~one hop; attribute that
# hop's share of the turn's tokens (total / hop-count), priced under the model it ran on.
_price_map = {r["dep"]: (float(r["in_price"]), float(r["out_price"])) for r in _pricing.collect()}
_td_turns = 0
_td_saving = 0.0
for _r in turns.select("agent_path", "input_tokens", "output_tokens", "model_deployment").collect():
    _parts = [p.strip() for p in str(_r["agent_path"] or "").split(",") if p.strip()]
    if any(_parts[i] == _parts[i + 1] and _parts[i] != "supervisor" for i in range(len(_parts) - 1)):
        _td_turns += 1
        _hops = max(len(_parts), 1)
        _pin, _pout = _price_map.get(_r["model_deployment"], (b_in, b_out))
        _td_saving += ((float(_r["input_tokens"] or 0) * _pin
                        + float(_r["output_tokens"] or 0) * _pout) / _hops) / 1e6
_td_saving = round(_td_saving, 6)

# Model-selection opportunity parity with build_model_selection_recommendation:
# tenant-scoped premium deployments with short output are downgrade candidates. Re-price
# only their observed tokens from the gpt-5.1 baseline to gpt-5-nano.
_MS_PREMIUM = {"gpt-5.1", "gpt-5"}
_MS_SHORT_OUTPUT_MAX = 250
_nano_in, _nano_out = _price_map.get("gpt-5-nano", (0.05, 0.40))
_ms_candidates = 0
_ms_candidate_in = 0
_ms_candidate_out = 0
_ms_models = {}
for _r in turns.select("model_deployment", "input_tokens", "output_tokens").collect():
    _deployment = str(_r["model_deployment"] or "Unknown")
    _ms_models[_deployment] = _ms_models.get(_deployment, 0) + 1
    _output_tokens = int(_r["output_tokens"] or 0)
    if _deployment in _MS_PREMIUM and _output_tokens < _MS_SHORT_OUTPUT_MAX:
        _ms_candidates += 1
        _ms_candidate_in += int(_r["input_tokens"] or 0)
        _ms_candidate_out += _output_tokens
_ms_cost_now = (_ms_candidate_in * b_in + _ms_candidate_out * b_out) / 1e6
_ms_cost_proposed = (_ms_candidate_in * _nano_in + _ms_candidate_out * _nano_out) / 1e6
_ms_saving = round(_ms_cost_now - _ms_cost_proposed, 6)

# The detected issues THIS data supports, each with its ENGINE-computed saving (guardrail #3
# makes the engine number authoritative; the analyst may argue but cannot change it). The
# loop below mirrors pipeline.analyze — one card per detection:
#   (1) tenant premium-short-output downgrade candidates -> gpt-5.1-to-nano re-pricing, seam=config
#   (2) repeated-node structural -> the avoidable duplicated hop priced above, seam=prompt (L3)
_detections = [
    {"detector": "counterfactual.model_fit", "kind": "counterfactual", "agent": "supervisor",
     "dimension": "model selection \u00b7 cost", "opportunity_id": "opp-modelfit-supervisor",
     "scenario": "model-selection", "title": "Capability-tiered model selection (discovered)",
     "evidence": {"total_turns": _n, "downgrade_candidates": _ms_candidates,
                  "downgrade_pct": round(100 * _ms_candidates / max(_n, 1), 1),
                  "model_distribution": _ms_models},
     "engine_saving": _ms_saving},
    {"detector": "structural.repeated_node", "kind": "structural", "agent": "find_places",
     "dimension": "workflow efficiency \u00b7 tool use", "opportunity_id": "opp-repeated-node",
     "scenario": "tool-call-dedup",
     "title": "Redundant tool calls \u2014 supervisor prompt fix (discovered)",
     "evidence": {"redundant_tool_turns": _td_turns, "estimated_saving_usd": _td_saving},
     "engine_saving": _td_saving},
]


def _analyst_prompt(det):
    return ("Detected issue:\n"
            f"  detector: {det['detector']}\n  kind: {det['kind']}\n  agent: {det['agent']}\n"
            f"  dimension: {det['dimension']}\n  opportunity_id: {det['opportunity_id']}\n"
            f"  evidence: {_json.dumps(det['evidence'])}\n\n"
            "Allowed targets by seam:\n"
            f"  config: {SURFACE['config']}\n  prompt: {SURFACE['prompt']}\n  code: {SURFACE['code']}\n"
            "Sample trace ids you may cite: ['trace-1','trace-2']")


def _aad_token():
    try:
        import notebookutils as _nb
    except Exception:
        import mssparkutils as _nb
    for aud in ("https://cognitiveservices.azure.com", "pbi"):
        try:
            tk = _nb.credentials.getToken(aud)
            if tk:
                return tk
        except Exception:
            pass
    raise RuntimeError("no Entra token for Azure OpenAI")


def _call_llm(system, user):
    url = (f"{AOAI_ENDPOINT.rstrip('/')}/openai/deployments/{AOAI_DEPLOYMENT}"
           f"/chat/completions?api-version={AOAI_API_VERSION}")
    r = _rq.post(url, headers={"Authorization": f"Bearer {_aad_token()}",
                               "Content-Type": "application/json"},
                 json={"messages": [{"role": "system", "content": system},
                                    {"role": "user", "content": user}]}, timeout=90)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def _parse_card(text, det):
    t = (text or "").strip()
    if t.startswith("```"):
        t = t.strip("`")
        t = t[t.find("{"):]
    try:
        obj = _json.loads(t[t.find("{"): t.rfind("}") + 1])
    except Exception:
        return None
    return {"agent": det["agent"], "dimension": det["dimension"],
            "seam": str(obj.get("seam", "")), "target": str(obj.get("target", "")),
            "evidence": obj.get("evidence") or [],
            "claimed_saving": float(obj.get("claimed_saving", 0) or 0),
            "apply_mode": str(obj.get("apply_mode", "")),
            "autonomy_ceiling": str(obj.get("autonomy_ceiling", ""))}


def _default_card(det):   # deterministic fallback (engine parity) when the LLM is unavailable
    seam, target = OPPORTUNITY_SEAMS.get(det["opportunity_id"], ("prompt", "supervisor.prompty"))
    return {"agent": det["agent"], "dimension": det["dimension"], "seam": seam,
            "target": target, "claimed_saving": 0.0, "apply_mode": "", "autonomy_ceiling": "",
            "evidence": [{"detector": det["detector"], "opportunity_id": det["opportunity_id"],
                          "traces": ["sample-trace"]}]}


def _guardrail(card, det):   # the deterministic rules; returns (normalized|None, why)
    if card["seam"] not in SURFACE:
        return None, f"reject: unknown seam '{card['seam']}'"
    if card["target"] not in SURFACE[card["seam"]]:
        return None, f"reject: target '{card['target']}' off the declared {card['seam']} surface"
    _expected = OPPORTUNITY_SEAMS.get(det["opportunity_id"])
    if _expected is None:
        return None, f"reject: no declared seam for opportunity '{det['opportunity_id']}'"
    if (card["seam"], card["target"]) != _expected:
        return None, (
            f"reject: opportunity '{det['opportunity_id']}' requires "
            f"{_expected[0]} -> {_expected[1]}"
        )
    if not card["evidence"]:
        return None, "reject: uncited"
    for e in card["evidence"]:
        if not (e.get("detector") and e.get("opportunity_id") and e.get("traces")):
            return None, "reject: evidence missing detector/opportunity_id/traces"
    return {"agent": card["agent"], "dimension": card["dimension"], "seam": card["seam"],
            "target": card["target"], "saving": det["engine_saving"],
            "apply_mode": SEAM_APPLY_MODE[card["seam"]],
            "autonomy_ceiling": SEAM_CEILING[card["seam"]]}, "accepted (engine-computed saving; LLM $ ignored)"


# propose -> parse -> (fallback) -> guardrail, once PER detected opportunity (pipeline.analyze's loop)
def _propose(det):
    if AOAI_ENDPOINT:
        try:
            return _parse_card(_call_llm(SYSTEM, _analyst_prompt(det)), det) or _default_card(det)
        except Exception as _e:
            print("LLM analyst unavailable, using deterministic fallback:", str(_e)[:160])
            return _default_card(det)
    print("AOAI_ENDPOINT blank -> deterministic fallback proposer for", det["opportunity_id"])
    return _default_card(det)


from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType


# Flat evidence/caveat for the BI cards: the nested `card` object does NOT surface as columns
# over the Fabric mirror, so we project a compact one-line proof + a caveat onto each row.
# Mirrors summarize_card_evidence / card_caveat in
# src/app/services/optimization_recommendations.py so these analyst cards read identically to
# the app-plane cards they supersede (no regression when the notebook runs last).
def _evidence_line(_det):
    _s = _det["scenario"]
    if _s == "model-selection":
        return (f"{_n:,} turns \u00b7 {_ms_candidates:,} downgrade candidates "
                f"({round(100 * _ms_candidates / max(_n, 1), 1)}%) \u00b7 "
                f"{len(_ms_models)} models")
    if _s == "tool-call-dedup":
        return f"{_td_turns:,} redundant tool turns of {_n:,}"
    return ""


def _caveat_line(_det):
    _s = _det["scenario"]
    if _s == "model-selection":
        return ("Estimate only \u2014 tenant downgrade candidates re-priced from gpt-5.1 to "
                "gpt-5-nano using their observed input/output tokens; measured before/after "
                "remains authoritative.")
    if _s == "tool-call-dedup":
        return ("Turn-grain estimate \u2014 one avoidable duplicated hop per repeated-node turn, "
                "priced under the model it ran on.")
    return ""


_policy_status = {}
if all(_c in policies.columns for _c in ("scenario", "status")):
    _policy_status = {
        _r["scenario"]: _r["status"]
        for _r in policies.select("scenario", "status").collect()
        if _r["scenario"]
    }

_slo = {"slo": 4.0, "min_confidence": 0.7, "min_effect": 0.05, "by": "default"}
if "type" in governance.columns:
    _slo_query = governance.where(F.col("type") == "slo_policy")
    if "timeStamp" in governance.columns:
        _slo_query = _slo_query.orderBy(F.col("timeStamp").desc())
    _slo_docs = _slo_query.limit(1).collect()
    if _slo_docs:
        _slo_doc = _slo_docs[0].asDict()
        for _key in _slo:
            if _slo_doc.get(_key) is not None:
                _slo[_key] = _slo_doc[_key]

def _has_positive_signal(_det):
    if _det["scenario"] == "model-selection":
        return _ms_candidates > 0 and float(_det["engine_saving"]) > 0
    if _det["scenario"] == "tool-call-dedup":
        return _td_turns > 0 and float(_det["engine_saving"]) > 0
    return False


def _suppression_reason(_det):
    if _det["scenario"] == "model-selection":
        return (
            f"no positive model-selection signal: downgrade_candidates={_ms_candidates}, "
            f"estimated_saving_usd={float(_det['engine_saving']):.6f}"
        )
    if _det["scenario"] == "tool-call-dedup":
        return (
            f"no positive tool-call-dedup signal: redundant_tool_turns={_td_turns}, "
            f"estimated_saving_usd={float(_det['engine_saving']):.6f}"
        )
    return "unsupported opportunity"


_positive_detections = [_det for _det in _detections if _has_positive_signal(_det)]
_suppressed_detections = [_det for _det in _detections if not _has_positive_signal(_det)]
_disc_rows, _agent_opp_rows, _rec_rows, _suppressed_rows = [], [], [], []
_total_spend = sum(float(_n.get("cost") or 0.0) for _n in _nodes)
for _det in _suppressed_detections:
    _reason = _suppression_reason(_det)
    for _id in (
        f"disc:{TENANT}:{_det['opportunity_id']}",
        f"agentopp::{TENANT}::{_det['opportunity_id']}",
        f"reccard::{TENANT}::{_det['scenario']}",
    ):
        _suppressed_rows.append(
            (_id, "suppressed_opportunity", TENANT, _det["scenario"], _reason, now)
        )
    print(f"suppressed analyst opportunity [{_det['opportunity_id']}]:", _reason)

for _rank, _det in enumerate(_positive_detections):
    _norm, _why = _guardrail(_propose(_det), _det)
    if _norm is None:                          # a bad LLM proposal -> fall back and guardrail that
        print("guardrail rejected the LLM card:", _why)
        _norm, _why = _guardrail(_default_card(_det), _det)
    print(f"analyst card [{_det['opportunity_id']}]:", _json.dumps(_norm), "->", _why)
    _disc_rows.append(
        (f"disc:{TENANT}:{_det['opportunity_id']}", "discovered_opportunity", TENANT, _rank,
         _det["opportunity_id"], _det["kind"], _norm["agent"], _norm["dimension"],
         _norm["seam"], _norm["target"], float(_norm["saving"]), _norm["apply_mode"],
         _norm["autonomy_ceiling"], _json.dumps(_det["evidence"]), now))
    _scenario_status = (
        _policy_status.get(_det["scenario"], "not_proposed")
        if _norm["apply_mode"] == "auto"
        else "proposed"
    )
    _display_state = (
        "Active" if _scenario_status == "active"
        else "Not applied" if _norm["apply_mode"] == "auto"
        else "Proposed"
    )
    _effect_pct = 100 * float(_norm["saving"]) / _total_spend if _total_spend else 0.0
    _agent_opp_rows.append(
        (f"agentopp::{TENANT}::{_det['opportunity_id']}", "agent_opportunity", TENANT,
         _norm["agent"], _norm["dimension"], _rank + 1,
         f"{_norm['seam']} \u2192 {_norm['target']}", float(_norm["saving"]), round(_effect_pct, 2),
         "Automatic" if _norm["apply_mode"] == "auto" else "Manual",
         _norm["autonomy_ceiling"], "\u2713" if _effect_pct / 100 >= float(_slo["min_effect"]) else "\u00d7",
         _display_state, now))
    _card_obj = {"scenario": _det["scenario"], "scenario_id": _det["scenario"], "title": _det["title"],
                 "dimension": _norm["dimension"], "apply_mode": _norm["apply_mode"],
                 "maturity": "discovered by the LLM analyst (engine-guardrailed)",
                 "estimated_saving_usd": float(_norm["saving"]), "status": _scenario_status}
    _rec_rows.append(
        (f"reccard::{TENANT}::{_det['scenario']}", "recommendation_card", TENANT, _det["scenario"],
         _det["scenario"], _rank, f"{_rank + 1} \u00b7 {_card_obj['title']}", _card_obj["title"],
         _card_obj["dimension"], _card_obj["apply_mode"], _card_obj["status"], _card_obj["maturity"],
         float(_card_obj["estimated_saving_usd"]),
         _evidence_line(_det), _caveat_line(_det), _card_obj, now))

_slo_rows = [
    (f"slometric::{TENANT}::1", "slo_metric", TENANT, 1, "1 \u00b7 Quality gate (e2e_quality \u2265)",
     f"{float(_slo['slo']):g}", now),
    (f"slometric::{TENANT}::2", "slo_metric", TENANT, 2, "2 \u00b7 Min confidence",
     f"{100 * float(_slo['min_confidence']):.1f}%", now),
    (f"slometric::{TENANT}::3", "slo_metric", TENANT, 3, "3 \u00b7 Min effect",
     f"{100 * float(_slo['min_effect']):.1f}%", now),
    (f"slometric::{TENANT}::4", "slo_metric", TENANT, 4, "4 \u00b7 Source", str(_slo["by"]), now),
]
slo_df = spark.createDataFrame(
    _slo_rows, ["id", "type", "tenantId", "order", "title", "evidence_line", "computed_at"])

_rec_schema = StructType([
    StructField("id", StringType()), StructField("type", StringType()), StructField("tenantId", StringType()),
    StructField("scenario", StringType()), StructField("scenario_id", StringType()),
    StructField("order", LongType()), StructField("note", StringType()), StructField("title", StringType()),
    StructField("dimension", StringType()), StructField("apply_mode", StringType()),
    StructField("status", StringType()), StructField("maturity", StringType()),
    StructField("estimated_saving_usd", DoubleType()),
    StructField("evidence_line", StringType()), StructField("caveat", StringType()),
    StructField("card", StructType([
        StructField("scenario", StringType()), StructField("scenario_id", StringType()),
        StructField("title", StringType()), StructField("dimension", StringType()),
        StructField("apply_mode", StringType()), StructField("maturity", StringType()),
        StructField("estimated_saving_usd", DoubleType()), StructField("status", StringType())])),
    StructField("computed_at", StringType()),
])

# reverse-ETL positive analyst/report rows only. Suppressed detections overwrite the same
# deterministic IDs with a non-report type so a disappeared signal cannot leave stale cards.
_write_dfs = [slo_df]
if _disc_rows:
    _write_dfs.append(spark.createDataFrame(
        _disc_rows,
        ["id", "type", "tenantId", "rank", "opportunity_id", "kind", "agent", "dimension", "seam",
         "target", "saving", "apply_mode", "autonomy_ceiling", "evidence_json", "computed_at"]))
if _agent_opp_rows:
    _write_dfs.append(spark.createDataFrame(
        _agent_opp_rows,
        ["id", "type", "tenantId", "agent", "dimension", "order", "note", "saving_usd",
         "saving_pct", "apply_mode", "maturity", "method", "status", "computed_at"]))
if _rec_rows:
    _write_dfs.append(spark.createDataFrame(_rec_rows, _rec_schema))
if _suppressed_rows:
    _write_dfs.append(spark.createDataFrame(
        _suppressed_rows,
        ["id", "type", "tenantId", "scenario", "reason", "computed_at"]))

for _df in _write_dfs:
    _df.write.format("cosmos.oltp").options(**cosmos_write).mode("append").save()
print(f"Analyst reverse-ETL complete -> {len(_disc_rows)} discovered_opportunity + "
      f"{len(_agent_opp_rows)} agent_opportunity + {len(_slo_rows)} slo_metric + "
      f"{len(_rec_rows)} recommendation_card + {len(_suppressed_rows)} suppressed rows ("
      + ", ".join(f"{d['scenario']} ${d['engine_saving']}" for d in _positive_detections) + ")")

In [ ]:
_checkpoint("complete")